# مختبر بحث الإشارات (Signal Discovery Lab)

**الهدف:** بنية تحتية لاكتشاف مرشّحين جدد للإشارة (`DISCOVERY_TRACKS` في
`signal_evaluation_axis`: `hypothesis_driven`, `data_driven`,
`literature_mining`, `genetic_search`) — **بلا أي تدريب شبكة عصبية**، فقط
دوال جاهزة/رخيصة (مؤشرات موجودة أصلاً في `feature_order`، أو انحدار خطّي
Ridge يُحسَب في أجزاء من الثانية لكل نافذة) مُقيَّمة عبر نفس صرامة المحور
(IC + عُشر + خطّ أساس عشوائي عبر نوافذ متحرّكة، كما في H001/H002).

**لماذا بلا تدريب؟** كل تجربة NIG-TimeNet v2 حتى الآن (main.ipynb، H002)
استهلكت وقتاً وتكلفة حوسبة حقيقية لكل نافذة/تشغيل. هذا الدفتر يفحص عشرات
المرشّحين دفعة واحدة **بتكلفة تقارب الصفر** (ثوانٍ لا دقائق) قبل تبرير أي
تدريب فعلي — فرز أوّلي رخيص، لا بديل عن H001/H002 حين يستحقّ مرشّح تدريباً حقيقياً.

**⚠️ تحذير حرِج — اقرأه قبل الوثوق بأي نتيجة على `high`/`low`:**
اكتُشف أثناء بناء هذا الدفتر أن `y_high_reg`/`y_low_reg` (`reg_target_mode=
'return'`) تُقارَنان بمرجع "نفس النوع" (`last_high`/`last_low`) لا
`last_close` (راجع تنبيه رقم ٢٠ في رأس `crypto_data_pipeline_v6.ipynb`
للتفاصيل والبرهان الرقمي الكامل). هذا يجعل ميزات شكل الشمعة الأخيرة
(`BODY_ratio`, `WICK_upper/lower`, وبدرجة أقل `RET_1`) تُظهر ارتباطاً زائفاً
**قوياً جداً** (سبيرمان ≈+0.51 على بيانات حقيقية) بهذين الهدفين تحديداً —
اختفى تماماً (إلى ≈+0.01) عند توحيد المرجع. **كل دالة تقييم في هذا الدفتر
تستخدم `clean_reg_target` (مرجع `last_close` موحّد) تلقائياً لـ`high`/`low`
— لا `y_high_reg`/`y_low_reg` الأصليين مباشرة.** `close_reg` غير متأثر أصلاً
(مرجعه `last_close` دائماً).

## ١) التجهيز — تحميل تعريفات الدفاتر بأمان (بلا تنفيذ تلقائي لخلايا الأمثلة)

`%run` مباشر لـ`signal_evaluation_axis` قد يُنفِّذ خلايا أمثلته (تفترض
`dataset`/`windows` جاهزين من جلسة سابقة) فيفشل بخطأ متغيّر غير معرَّف. نفس
الأسلوب المُستخدَم فعلاً لاختبار H002 على بيانات حقيقية: استخراج تعريفات
الدوال/الأصناف فقط عبر `ast`، بلا كود سائق.

In [ ]:
# @title
!git clone -q https://github.com/yuosef772424/crypto-signal-prediction.git 2>/dev/null || true
%cd /content/crypto-signal-prediction

import json, ast, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd


def _notebook_code(path):
    nb = json.load(open(path, encoding="utf-8"))
    return "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")


def load_notebook_defs(path):
    """يستخرج تعريفات الدوال/الأصناف والاستيرادات والقيم الحرفية فقط من دفتر
    — يتجاهل خلايا الأمثلة/السائقة (متغيّرات تفاعلية غير معرَّفة، أو استدعاءات
    شبكية حقيقية). آمن لتحميل signal_evaluation_axis دون تشغيله بالكامل."""
    code_text = _notebook_code(path)
    code_text = "\n".join(l for l in code_text.split("\n")
                          if not l.strip().startswith(("%", "!")))
    tree = ast.parse(code_text)
    keep_types = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom)
    literal_types = (ast.Tuple, ast.List, ast.Constant, ast.Dict, ast.Set)
    kept, futures = [], []
    for node in tree.body:
        if isinstance(node, ast.ImportFrom) and node.module == "__future__":
            futures.append(node)
        elif isinstance(node, keep_types):
            kept.append(node)
        elif isinstance(node, ast.Assign) and isinstance(node.value, literal_types):
            kept.append(node)
    mod = ast.Module(body=futures[:1] + kept, type_ignores=[])
    ast.fix_missing_locations(mod)
    return ast.unparse(mod)


%run "crypto_data_pipeline_v6.ipynb"

exec(compile(load_notebook_defs('signal_evaluation_axis (3).ipynb'), "axis", "exec"))
print("✅ تعريفات المحور مُحمَّلة: rolling_splits, evaluate_windows, concat_splits, "
      "extract_actuals, register_hypothesis, list_registry")

## ٢) تحميل البيانات وبناء النوافذ المتحرّكة

نفس الإعداد المُستخدَم في H002 — عدّله حسب مجموعة أصولك.

In [ ]:
# @title
dataset = load_data_from_drive()  # أو مسار preprocessing_output_latest.pkl.gz لديك
FEATURE_ORDER = dataset["feature_order"]

update_config({"min_split_samples": 10})  # ⚠️ خفّضه فقط إن أصولك القليلة تحتاجه (راجع H002)
windows = rolling_splits(
    dataset, test_span="30D", val_span="15D", initial_train_span="365D",
    step="30D", max_windows=12, keep_asset_test_separate=False, config=CONFIG,
)

## ٣) الحارس ضدّ أثر مرجع "نفس النوع" — `clean_reg_target`

استخدمه دائماً بدل `y_high_reg`/`y_low_reg` الأصليين عند تقييم أي مرشّح جديد
ضد high/low (راجع التحذير أعلى الدفتر). `close_reg` غير متأثر فيبقى كما هو.

In [ ]:
# @title
def clean_reg_target(split, target):
    """`y_{target}_reg` بمرجع `last_close` موحّد لكل الأهداف — لا
    `last_high`/`last_low` الأصليين (own-kind reference) اللذين يحملان أثر
    شكل الشمعة الأخيرة (راجع تنبيه رقم ٢٠ في crypto_data_pipeline_v6). لهدف
    `close` يعادل `y_close_reg` تماماً (نفس المرجع أصلاً) فيُقرَأ منه مباشرة."""
    if target == "close":
        return extract_actuals(split, target_key="y_close_reg")
    split = concat_splits(split)
    lc = np.asarray(split["last_candles"])
    last_close = lc[:, LAST_COLUMNS.index("last_close")]
    future_col = {"high": "future_high_max", "low": "future_low_min"}[target]
    future = lc[:, LAST_COLUMNS.index(future_col)]
    return (future - last_close) / last_close


def evaluate_candidate(predict_fn, target, windows, n_shuffles=1000, min_samples=10, seed=42, verbose=False):
    """يقيّم مرشّحاً واحداً عبر كل النوافذ — بديل رقيق لـ
    evaluate_hypothesis_over_rolling_windows يستخدم دائماً clean_reg_target
    (لا target_key خام) فلا يُمكن نسيان الحارس بالخطأ."""
    names = [f"نافذة {i + 1}" for i in range(len(windows))]
    results = []
    for name, (train, val, test) in zip(names, windows):
        test_flat = concat_splits(test)
        preds = np.asarray(predict_fn(train, val, test), dtype="float64")
        actuals = clean_reg_target(test_flat, target)
        if len(preds) != len(actuals):
            raise ValueError(f"[{name}] طول التنبؤات ({len(preds)}) ≠ طول الأهداف ({len(actuals)}).")
        results.append((name, preds, actuals))
    return evaluate_windows(results, n_shuffles=n_shuffles, min_samples=min_samples, seed=seed, verbose=verbose)

## ٤) إطار المرشّح الواحد — أي ميزة جاهزة كمرشّح فوراً

`make_feature_predict_fn` يحوّل أي عمود من `feature_order` (بعد تحويل اختياري
— عكس، تمركز حول نقطة، إلخ) إلى `predict_fn` جاهزة لـ`evaluate_candidate`،
بنفس نمط `momentum_predict_fn` في المحور لكن مُعمَّمة لأي ميزة.

In [ ]:
# @title
def extract_feature_last_value(split, feature, tf=None, feature_order=None):
    """آخر قيمة (خطوة زمنية أخيرة) لميزة واحدة داخل نافذة كل عيّنة — حالة
    المؤشر عند لحظة القرار، لا فرقها كـextract_feature_last_diff."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    idx = feature_order.index(feature)
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])
    return X[:, -1, idx]


def extract_feature_matrix(split, features=None, tf=None, feature_order=None):
    """مصفوفة كل الميزات (أو مجموعة فرعية) في آخر خطوة زمنية — (N, len(features)).
    أعمّ من extract_feature_last_value: تُستخدَم لأي مرشّح يحتاج أكثر من ميزة
    معاً (تفاعل، مركَّب Ridge، تدريب Isolation Forest، أو دالة مخصَّصة كاملة
    على متجه الميزات — kind='custom'/'trained' أدناه)."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])[:, -1, :]
    if features is not None:
        idx = [feature_order.index(f) for f in features]
        X = X[:, idx]
    return X


def extract_feature_series(split, feature, tf=None, feature_order=None):
    """كل الخطوات الزمنية لميزة واحدة داخل نافذة كل عيّنة — (N, T)، لا آخر
    خطوة فقط كـextract_feature_last_value. لازمة لأي مؤشر مشتقّ يحتاج
    تاريخاً كاملاً ضمن النافذة (CMO/TSI عبر pandas_ta مثلاً، لا يُحسَبان من
    قيمة أخيرة وحدها) — kind='series' أدناه."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    idx = feature_order.index(feature)
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])
    return X[:, :, idx]


def make_feature_predict_fn(feature, transform=None, tf=None, feature_order=None):
    """predict_fn جاهزة لـevaluate_candidate من أي ميزة في feature_order.
    مرّر transform (مثلاً lambda v: -(v-50.0) لعكس RSI حول نقطة المنتصف)
    لصياغة فرضية اتجاه محدّدة بدل القيمة الخام."""
    def predict_fn(train, val, test):
        v = extract_feature_last_value(test, feature=feature, tf=tf, feature_order=feature_order)
        return transform(v) if transform else v
    return predict_fn


def make_interaction_predict_fn(feat_a, feat_b, op="mul", transform=None, tf=None, feature_order=None):
    """predict_fn من تفاعل بين ميزتين موجودتين (ضرب أو فرق) — يغطي فرضيات
    "تفاعلات" (قسم ٣ في خطة المشروع، مثال: حجم×جسم الشمعة) بلا الحاجة لعمود
    ميزة جديد محسوب مسبقاً في خط الأنابيب."""
    def predict_fn(train, val, test):
        a = extract_feature_last_value(test, feature=feat_a, tf=tf, feature_order=feature_order)
        b = extract_feature_last_value(test, feature=feat_b, tf=tf, feature_order=feature_order)
        v = a * b if op == "mul" else (a - b)
        return transform(v) if transform else v
    return predict_fn


def make_custom_predict_fn(fn, tf=None, feature_order=None):
    """predict_fn من دالة مخصَّصة على متجه الميزات الكامل عند آخر خطوة زمنية
    (`fn(X_last, feature_order) -> np.ndarray`) — بلا تدريب (test فقط، بلا
    استخدام train/val). يغطي فرضيات "توليد ممنهج" بلا حاجة لعمود ميزة/تفاعل
    ثنائي جاهز: مقلوب الافتراض الضمني (مثلاً RSI مطبَّع بالتقلب)، كسر
    التناظر (معادلتان مختلفتان للصعود/الهبوط عمداً)، إلخ — راجع قسم "توليد
    فرضيات ممنهج" أدناه."""
    def predict_fn(train, val, test):
        X_last = extract_feature_matrix(test, tf=tf, feature_order=feature_order)
        return fn(X_last, feature_order)
    return predict_fn


def make_series_predict_fn(fn, feature="close", tf=None, feature_order=None):
    """predict_fn من دالة مخصَّصة تُطبَّق على كامل نافذة ميزة واحدة عبر
    الزمن (`fn(series_2d) -> np.ndarray`، حيث `series_2d` شكلها (N, T)) —
    بخلاف kind='custom' (متجه كل الميزات معاً، لكن آخر خطوة فقط)، هذا لميزة
    واحدة تحتاج تاريخاً كاملاً ضمن النافذة (مؤشر مُشتقّ عبر pandas_ta مثل
    CMO/TSI لا يُحسَب من قيمة أخيرة وحدها)."""
    def predict_fn(train, val, test):
        series = extract_feature_series(test, feature=feature, tf=tf, feature_order=feature_order)
        return fn(series)
    return predict_fn


def make_candidate_predict_fn(cand, tf=None, feature_order=None):
    """يبني predict_fn من قاموس مرشّح واحد بصرف النظر عن نوعه (`kind`) —
    نقطة التفرّع الوحيدة، فلا يحتاج scan_candidates/run_batch_and_register
    معرفة الفرق بين الأنواع:

    * الافتراضي (بلا `kind`): ميزة واحدة عبر make_feature_predict_fn.
    * `kind="interaction"`: تفاعل بين ميزتين (`feat_a`/`feat_b`/`op`).
    * `kind="custom"`: دالة مخصَّصة بلا تدريب على متجه الميزات الكامل (`fn`).
    * `kind="trained"`: مرشّح يحتاج تدريباً على train (مثل Isolation Forest) —
      `builder(feature_order=...)` يُرجع predict_fn جاهزة (نفس نمط
      make_ridge_composite_predict_fn في قسم البحث التركيبي).
    * `kind="series"`: دالة مخصَّصة على كامل نافذة ميزة واحدة عبر الزمن
      (`fn`/`feature`) — لمؤشر مشتقّ يحتاج تاريخاً كاملاً (CMO/TSI)."""
    kind = cand.get("kind", "feature")
    if kind == "interaction":
        return make_interaction_predict_fn(cand["feat_a"], cand["feat_b"], op=cand.get("op", "mul"),
                                           transform=cand.get("transform"), tf=tf, feature_order=feature_order)
    if kind == "custom":
        return make_custom_predict_fn(cand["fn"], tf=tf, feature_order=feature_order)
    if kind == "trained":
        return cand["builder"](feature_order=feature_order)
    if kind == "series":
        return make_series_predict_fn(cand["fn"], feature=cand.get("feature", "close"), tf=tf, feature_order=feature_order)
    return make_feature_predict_fn(cand["feature"], transform=cand.get("transform"), tf=tf, feature_order=feature_order)

## ٥) مكتبة مرشّحين جاهزين (`literature_mining` + `data_driven`)

كل مرشّح: اسم، مسار اكتشاف (`DISCOVERY_TRACKS`)، ميزة من `feature_order`،
وتحويل اختياري يصيغ فرضية اتجاه (ارتداد/استمرار). أضِف مرشّحين جدداً بنفس
الشكل — لا حاجة لتعديل أي دالة أخرى.

In [ ]:
# @title
CANDIDATE_SIGNALS = [
    {"name": "RSI_14_reversion", "track": "literature_mining", "feature": "RSI_14",
     "transform": lambda v: -(v - 50.0), "hypothesis": "RSI متطرف يعكس (mean-reversion كلاسيكي)"},
    {"name": "MACDh_12_26_9_momentum", "track": "literature_mining", "feature": "MACDh_12_26_9",
     "transform": None, "hypothesis": "زخم MACD histogram يستمر"},
    {"name": "BBP_reversion", "track": "literature_mining", "feature": "BBP_20_2.0",
     "transform": lambda v: -(v - 0.5), "hypothesis": "موقع متطرف ضمن نطاق بولنجر يعكس"},
    {"name": "STOCH_reversion", "track": "literature_mining", "feature": "STOCHk_14_3_3",
     "transform": lambda v: -(v - 50.0), "hypothesis": "ستوكاستك متطرف يعكس"},
    {"name": "MFI_reversion", "track": "literature_mining", "feature": "MFI_14",
     "transform": lambda v: -(v - 50.0), "hypothesis": "تدفّق نقدي متطرف يعكس"},
    {"name": "CMF_momentum", "track": "data_driven", "feature": "CMF_20",
     "transform": None, "hypothesis": "تدفّق نقدي موجب يستمر"},
    {"name": "NATR_neg_vol", "track": "data_driven", "feature": "NATR_14",
     "transform": lambda v: -v, "hypothesis": "تقلّب مرتفع يسبق عائداً سالباً"},
    {"name": "ADX_trend_strength", "track": "data_driven", "feature": "ADX_14",
     "transform": None, "hypothesis": "قوة اتجاه مرتفعة تدعم استمراره"},
    {"name": "RET_1_reversion", "track": "hypothesis_driven", "feature": "RET_1",
     "transform": lambda v: -v, "hypothesis": "انعكاس قصير المدى (H001) بميزة RET_1 مباشرة"},
    {"name": "RET_6_momentum", "track": "literature_mining", "feature": "RET_6", "transform": None,
     "hypothesis": "زخم متوسط المدى (6 شموع)"},
    {"name": "RET_24_momentum", "track": "literature_mining", "feature": "RET_24", "transform": None,
     "hypothesis": "زخم أطول مدى (24 شمعة)"},
    {"name": "VOLZ_volume_shock", "track": "data_driven", "feature": "VOLZ_20", "transform": None,
     "hypothesis": "فورة حجم تسبق حركة سعرية"},
    {"name": "VOLR_regime", "track": "data_driven", "feature": "VOLR_6_24", "transform": None,
     "hypothesis": "نسبة تقلّب قصير/طويل المدى تكشف نظام سعري"},
    {"name": "POS_14_reversion", "track": "data_driven", "feature": "POS_14",
     "transform": lambda v: -(v - 0.5), "hypothesis": "موقع متطرف ضمن مدى 14 يعكس"},
    {"name": "MKT_beta", "track": "data_driven", "feature": "MKT_ret_1", "transform": None,
     "hypothesis": "عائد السوق العام (بيتا) يتنبأ بعائد الأصل — ⚠️ غير قابلة للاختبار فعلياً على أي "
                   "dataset بُني بـ market_context.enabled=False (الافتراضي): MKT_ret_1 تكون صفراً "
                   "حرفياً لكل عيّنة (تحقّق مباشر)، فـIC=0.0 دقيقاً لا يعني رفض الفرضية بدليل، بل "
                   "أنها لم تُختبَر بعد — راجع حاشية 'سياق عابر للأصول' في خطة المشروع"},
    {"name": "WICK_upper_rejection", "track": "literature_mining", "feature": "WICK_upper",
     "transform": lambda v: -v, "hypothesis": "ذيل علوي طويل إشارة رفض صعود"},
    {"name": "WICK_lower_rejection", "track": "literature_mining", "feature": "WICK_lower",
     "transform": None, "hypothesis": "ذيل سفلي طويل إشارة رفض هبوط"},
]
print(f"{len(CANDIDATE_SIGNALS)} مرشّحاً جاهزاً — أضِف المزيد بنفس الشكل أعلاه.")

### ملحق) مرشّحون إضافيون — تفاعلات + funding rate (المرحلتان ١-٢ من الخطة)

يغطي `EXPLORATORY_CANDIDATES` نمطين لم تغطِّهما `CANDIDATE_SIGNALS` أعلاه:

* **تفاعل بين ميزتين** (`kind="interaction"`، عبر `make_interaction_predict_fn`) —
  مثال "حجم×جسم الشمعة" من قسم "الفرضيات الاستكشافية" في الخطة: حجم غير
  عادي (`VOLZ_20`) مع جسم شمعة كبير (`BODY_ratio`) قد يعني دخول لاعب كبير،
  لا ضجيجاً عادياً — بضرب ميزتين موجودتين أصلاً، بلا عمود جديد في خط الأنابيب.
* **funding rate كمقياس تموضع متطرف** — الفرضية الثالثة في جدول "أساس" بالخطة
  (`FUND_rate_z`، مبنية أصلاً في خط الأنابيب عبر `add_funding_oi_features`
  لكن غير مُفعَّلة في `feature_order` بشكل افتراضي؛ راجع الخطوة التالية في
  README). المرشّح هنا جاهز لأي `dataset` يحمل هذه الميزة فعلياً — سيفشل
  بخطأ واضح (`ValueError`) على بيانات لا تحمل `FUND_rate_z`، لا صمتاً.

In [ ]:
# @title
EXPLORATORY_CANDIDATES = [
    {"name": "VOLZ_x_BODY", "track": "data_driven", "kind": "interaction",
     "feat_a": "VOLZ_20", "feat_b": "BODY_ratio", "op": "mul", "transform": None,
     "hypothesis": "حجم غير عادي × جسم شمعة كبير = دخول لاعب كبير، لا ضجيج عادي"},
    {"name": "FUND_rate_extreme_position", "track": "hypothesis_driven", "feature": "FUND_rate_z",
     "transform": lambda v: -v, "hypothesis": (
         "funding مرتفع جداً = طويلون مفرطون بالرافعة → خطر تصفية → انعكاس هبوطي محتمل "
         "(العكس لـfunding سالب جداً)")},
]
print(f"{len(EXPLORATORY_CANDIDATES)} مرشّحاً استكشافياً إضافياً — "
      "FUND_rate_extreme_position يحتاج dataset يحمل FUND_rate_z فعلياً.")

<cell_type>markdown</cell_type>### ملحق ٢) توليد فرضيات ممنهج — آليات الخطة الخمس، لا "جرّب مؤشراً جديداً"

قسم "طرق توليد فرضيات جديدة" في خطة المشروع يُلاحظ أن المؤشرات القياسية
استُهلكت (RSI، MACD، Bollinger...) لكن **آليات** توليد الفرضية أقلّ استهلاكاً
بكثير من الأدوات نفسها. `GENERATIVE_CANDIDATES` يجسّد ثلاثاً من الخمس بكود
حقيقي (لا وصفاً نظرياً)، كلّ واحدة عبر `kind="custom"`/`"trained"` الجديدين
في `make_candidate_predict_fn` أعلاه:

| الآلية (من الخطة) | المرشّح | الفكرة |
| --- | --- | --- |
| **مقلوب الافتراض الضمني** | `RSI_vol_adjusted_reversion` | RSI يفترض أن عتبتي 30/70 ثابتتان بصرف النظر عن حالة السوق — هنا يُطبَّع الانحراف عن 50 بالتقلب الحالي (`NATR_14`)، فتصير العتبة **نسبية لنظام التقلّب**، لا مطلقة |
| **كسر التناظر عمداً** | `Asymmetric_momentum_downside_weighted` | أغلب المؤشرات تعامل الصعود والهبوط بنفس المعادلة؛ سلوكياً الذعر يتحرّك أسرع من الجشع — هنا معادلتان مختلفتان فعلاً حسب الاتجاه: في الهبوط زخم قصير المدى (`RET_1`)، وفي الصعود زخم أبطأ (`RET_6`) (⚠️ ليس مجرّد تحجيم `RET_1` بوزن ثابت — ذلك تحويل رتيب لا يغيّر ترتيب سبيرمان، فيُعطي IC مطابقاً لـRET_1 خام تماماً؛ استخدام متغيّرين مختلفين حسب الحالة يكسر هذا الرتابة فعلاً) |
| **النقل من مجال مختلف** | `IsolationForest_anomaly_score` | تقنية كشف شذوذ (لا "مؤشر تداول" أصلاً) — Isolation Forest يُدرَّب على train لكل نافذة (`kind="trained"`، بنفس نمط المركَّب Ridge)، ودرجة الشذوذ على test هي التنبؤ: هل الشذوذ الإحصائي نفسه يحمل معلومة اتجاه؟ |

الآليتان المتبقّيتان في الخطة (**تقطير الرؤية المتأخّرة** — يحتاج هدفاً
جديداً كلياً لا مجرّد ميزة، مثال DPO/ATR الموثَّق في الخطة؛ و**تحليل الأخطاء
المنهجية** — يحتاج نموذجاً مُدرَّباً فعلياً لفحص أخطائه) تحتاجان بنية تتجاوز
نطاق "مرشّح بلا تدريب" لهذا الدفتر تحديداً — موثَّقتان هنا كخطوة تالية
صريحة، لا مُنفَّذتين قسراً بشكل مبتور.

In [ ]:
# @title
def _rsi_vol_adjusted(X_last, feature_order):
    """(RSI-50) مطبَّعة بالتقلب الحالي (NATR_14) — عتبة تطرف نسبية للنظام
    السعري، لا 30/70 الثابتة. سالبة (نفترض ارتداداً، لا استمراراً)."""
    rsi = X_last[:, feature_order.index("RSI_14")]
    natr = X_last[:, feature_order.index("NATR_14")]
    return -(rsi - 50.0) / (natr + 1e-6)


def _asymmetric_momentum(X_last, feature_order):
    """في الهبوط (RET_1<0) نعتمد الزخم القصير (RET_1) — استجابة سريعة؛ في
    الصعود نعتمد زخماً أبطأ (RET_6) — تأكيداً أبطأ (الذعر أسرع من الجشع).
    ⚠️ تنبيه للحذر عند تصميم مرشّحين مماثلين: تحجيم بسيط (`ret1 * وزن`، لو
    كان الوزن ثابتاً في كل شقّ) هو تحويل رتيب لـRET_1 عالمياً، وسبيرمان
    (المقياس المُستخدَم في IC هنا) لا يتأثر بتحويل رتيب — أي IC سيتطابق مع
    IC خام RET_1 تماماً رغم اختلاف الشكل. هنا نستخدم متغيّرين مختلفين حسب
    الحالة (تحويل غير رتيب)، فترتيبه لا يطابق ترتيب RET_1 ولا RET_6 وحدهما
    (تحقّق تجريبي على بيانات حقيقية: سبيرمان مع RET_1 ≈0.73، مع RET_6 ≈0.66
    — لا ±1.0 لأيّهما)."""
    ret1 = X_last[:, feature_order.index("RET_1")]
    ret6 = X_last[:, feature_order.index("RET_6")]
    return np.where(ret1 < 0, ret1, ret6)


def make_isolation_forest_predict_fn(features, feature_order=None, contamination=0.1, random_state=42):
    """`kind="trained"`: يُدرِّب Isolation Forest على train (مُطبَّع بمتوسط/
    انحراف train نفسه)، ثم يُرجع درجة الشذوذ (`decision_function`، أعلى =
    أقلّ شذوذاً) على test — نفس نمط make_ridge_composite_predict_fn تماماً،
    لكن بلا هدف (كشف شذوذ غير مُشرَف، لا انحدار)."""
    def predict_fn(train, val, test):
        from sklearn.ensemble import IsolationForest
        train_flat = concat_splits(train)
        Xtr = extract_feature_matrix(train_flat, features, feature_order=feature_order)
        mu, sigma = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-9
        model = IsolationForest(contamination=contamination, random_state=random_state, n_estimators=100)
        model.fit((Xtr - mu) / sigma)
        Xte = extract_feature_matrix(test, features, feature_order=feature_order)
        return model.decision_function((Xte - mu) / sigma)
    return predict_fn


GENERATIVE_CANDIDATES = [
    {"name": "RSI_vol_adjusted_reversion", "track": "hypothesis_driven", "kind": "custom",
     "fn": _rsi_vol_adjusted, "mechanism": "assumption_inversion",
     "hypothesis": "عتبات RSI الثابتة (30/70) لا تناسب كل نظام تقلّب — تطبيع الانحراف عن 50 "
                   "بالتقلّب (NATR) يلتقط تطرّفاً نسبياً حقيقياً، لا مطلقاً"},
    {"name": "Asymmetric_momentum_downside_weighted", "track": "hypothesis_driven", "kind": "custom",
     "fn": _asymmetric_momentum, "mechanism": "asymmetry_hunting",
     "hypothesis": "الذعر يتحرّك أسرع من الجشع — في الهبوط زخم قصير المدى (RET_1) أدقّ، وفي "
                   "الصعود زخم أبطأ (RET_6) أنسب؛ معادلتان مختلفتان عمداً حسب الاتجاه"},
    {"name": "IsolationForest_anomaly_score", "track": "data_driven", "kind": "trained",
     "builder": lambda feature_order: make_isolation_forest_predict_fn(
         [f for f in feature_order if f != "close"], feature_order=feature_order),
     "mechanism": "analogical_transfer",
     "hypothesis": "شذوذ إحصائي في متجه الميزات قد يعكس حدثاً حقيقياً (تصفية، خبر) يحمل معلومة "
                   "اتجاه، لا ضجيجاً محايداً"},
]
print(f"{len(GENERATIVE_CANDIDATES)} مرشّحات توليد ممنهج — راجع جدول الآليات أعلاه.")

### ملحق ٣) إعادة اختبار أقوى نتائج استكشاف سابق (باكتيست خام ← المحور الصارم)

شارك صاحب المشروع نتائج استكشاف قديم خاص به: `generate_signals` (٧٢ إشارة
شراء من مكتبة مؤشرات `pandas_ta_classic` كاملة تقريباً) اختُبرت فردياً عبر
باكتيست مباشر (ربح/نسبة نجاح/profit factor)، **بلا أي تصحيح لمشكلة
الاختبارات المتعددة** — بالضبط الفخّ الذي تحذّر منه الخطة في قسم "البحث
الموجَّه" (آلاف التوليفات، فبعضها سينجح صدفة حتماً). أقوى النتائج بمعيار
profit_factor: `CCI`≈1.52، `STOCH`/`KDJ`≈1.37 (مُختبَر أصلاً أعلاه كـ
`STOCH_reversion`)، `TSI`≈1.36، `AD`/`WCP`≈1.33، `CMO`≈1.31 — لكن نسبة
النجاح لمعظمها منخفضة جداً (~7%)، نمط مثير للريبة (رابح بصفقات نادرة جداً)
يستحقّ فحص IC/عُشر/خطّ أساس عشوائي لا الاكتفاء بربح إجمالي.

**قيد إعادة الاختبار هنا:** `feature_order` الحالي لا يحمل high/low/volume
الخام (فقط نِسَباً مشتقّة منها: `RANGE_rel`، `BODY_ratio`، `VOLZ`/`VOLR`)،
فمؤشرات تحتاجها مباشرة (`CCI`، `AD`/`OBV`، `WCP`، `UO`، `QSTICK`، Vortex)
**لا يمكن إعادة اختبارها من هذا الدفتر بلا تعديل خط الأنابيب** لإضافتها
كميزات خام — خطوة مؤجَّلة، بنفس منطق تأجيل funding/OI، لا تُنفَّذ إلا بطلب
صريح. فقط `CMO`، `TSI`، و`DPO` (يحتاجان `close` فقط، وهي مُتاحة عبر كل
خطوات النافذة الزمنية لا آخر خطوة فقط) قابلة لإعادة الاختبار الآن، عبر
`kind="series"` الجديد في `make_candidate_predict_fn` أعلاه.

**⚠️ ملاحظة أمانة تقنية (TSI):** `TSI(13,25)` الأصلي (بارامترات
`generate_signals`) لا يتقارب إطلاقاً ضمن `window_size=32` الحالي لخط
الأنابيب (يحتاج نحو ٥١ خطوة إحماء) — قيمته الأخيرة تبقى NaN طوال النافذة،
فتحقّقتُ تجريبياً وعدّلت البارامترات إلى `(5, 13, 5)` (نفس آلية التنعيم
المزدوج، أفق أقصر) لتتقارب ضمن ٣٢ خطوة — **ليست نفس المعايرة الأصلية
المُختبَرة في الباكتيست القديم**، بل تكيّف مضطرّ بسبب طول النافذة، موثَّق
هنا صراحة لا مخفيّاً.

**تحديث DPO — تجربة مباشرة إضافية من صاحب المشروع:** بخلاف "تقطير الرؤية
المتأخّرة" (DPO المُعاد رسمه، `centered=True`، الموثَّق أعلى الدفتر كهدف لا
كمدخل)، هذه تجربة بصيغة `centered=False` **غير المُعاد رسمها** (لا تسرّب
معلومات مستقبلية، قابلة للتنفيذ حيّاً كمدخل مباشر) — اختُبرت فعلياً على
BTC/1H: `DPO(14, centered=False)` أعطى profit_factor=0.89 (خاسر، 78 صفقة)،
بينما `DPO(2, centered=False)` أعطى profit_factor=1.14 (ربح صافٍ موجب، 166
صفقة، نجاح 65.66%) — تباين حادّ بمجرّد تغيير الطول من 14 إلى 2 يستحقّ فحصاً
عبر IC/عُشر/خطّ أساس عشوائي عبر أصول متعدّدة، لا الاكتفاء بنتيجة أصل واحد.
كلا الطولين مُضافان أدناه (`DPO_2_reversion`/`DPO_14_reversion`) للمقارنة
المباشرة، بصيغة عكسية (فرضية ارتداد — DPO متطرف سالب/موجب يعكس، بنفس منطق
استراتيجية صاحب المشروع الأصلية: شراء عند DPO سالب جداً، بيع عند موجب جداً).

In [ ]:
# @title
def _cmo_reversal(series_2d, length=14):
    """CMO (Chande Momentum Oscillator) عبر pandas_ta_classic على كل عيّنة
    على حدة — عكسه لصياغة فرضية ارتداد (CMO متطرف يعكس)، بنفس منطق
    RSI_14_reversion. من أقوى نتائج الاستكشاف القديم (profit_factor≈1.31
    في الباكتيست الخام، بلا تصحيح للاختبارات المتعددة هناك)."""
    import pandas_ta_classic as ta
    out = np.full(series_2d.shape[0], np.nan)
    for i in range(series_2d.shape[0]):
        cmo = ta.cmo(pd.Series(series_2d[i]), length=length)
        if cmo is not None and len(cmo):
            out[i] = cmo.iloc[-1]
    return -out


def _tsi_momentum(series_2d, fast=5, slow=13, signal=5):
    """TSI (True Strength Index) عبر pandas_ta_classic — زخم مزدوج التنعيم،
    بلا عكس (فرضية استمرار، لا ارتداد). بارامترات مُقصَّرة (5/13/5 بدل
    13/25/13 الأصلية في الباكتيست القديم) — راجع "ملحق ٣" أعلاه لسبب هذا
    التكيّف (`window_size=32` لا يكفي لتقارب TSI(13,25))."""
    import pandas_ta_classic as ta
    out = np.full(series_2d.shape[0], np.nan)
    for i in range(series_2d.shape[0]):
        tsi = ta.tsi(pd.Series(series_2d[i]), fast=fast, slow=slow, signal=signal)
        if tsi is not None and len(tsi):
            out[i] = tsi.iloc[-1, 0]
    return out


def make_dpo_reversion(length):
    """DPO (Detrended Price Oscillator) بصيغته `centered=False` — لا إزاحة
    للخلف، بلا تسرّب معلومات مستقبلية، قابل للتنفيذ حيّاً (بخلاف
    `centered=True` المُوثَّق أعلى الدفتر كمثال "تقطير الرؤية المتأخّرة"،
    ذاك يُستخدَم كهدف لا كمدخل). عكسه لصياغة فرضية ارتداد — بنفس منطق
    استراتيجية صاحب المشروع الأصلية (شراء عند DPO سالب جداً). تجربة مباشرة
    من صاحب المشروع على BTC/1H: طول=2 → profit_factor=1.14، طول=14 →
    profit_factor=0.89 (خاسر) — كلا الطولين هنا للمقارنة عبر IC الصارم."""
    import pandas_ta_classic as ta

    def _fn(series_2d):
        out = np.full(series_2d.shape[0], np.nan)
        for i in range(series_2d.shape[0]):
            dpo = ta.dpo(pd.Series(series_2d[i]), length=length, centered=False)
            if dpo is not None and len(dpo):
                out[i] = dpo.iloc[-1]
        return -out
    return _fn


LEGACY_BACKTEST_CANDIDATES = [
    {"name": "CMO_14_reversion", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": _cmo_reversal,
     "hypothesis": "CMO متطرف يعكس (من أقوى نتائج باكتيست خام سابق للمشروع، profit_factor≈1.31)"},
    {"name": "TSI_5_13_5_momentum", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": _tsi_momentum,
     "hypothesis": "زخم TSI مزدوج التنعيم يستمر (من أقوى نتائج باكتيست خام سابق، "
                   "profit_factor≈1.36 على معايرة 13/25 الأصلية — هنا معايرة أقصر 5/13/5)"},
    {"name": "DPO_2_reversion", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": make_dpo_reversion(2),
     "hypothesis": "DPO(2, centered=False) متطرف يعكس — تجربة مباشرة لصاحب المشروع "
                   "على BTC/1H أعطت profit_factor=1.14"},
    {"name": "DPO_14_reversion", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": make_dpo_reversion(14),
     "hypothesis": "DPO(14, centered=False) متطرف يعكس — نفس التجربة بطول أطول، "
                   "أعطت profit_factor=0.89 (خاسر) على BTC/1H — للمقارنة المباشرة مع DPO_2"},
]
print(f"{len(LEGACY_BACKTEST_CANDIDATES)} مرشّحاً من إعادة اختبار الاستكشاف القديم — "
      "الباقي (CCI/AD/OBV/WCP/UO/QSTICK/Vortex) يحتاج high/low/volume الخام في feature_order (مؤجَّل).")

### النتيجة الفعلية لإعادة اختبار الاستكشاف القديم (تشغيل حقيقي)

أُجري التقييم فعلياً على نفس بيانات H002 (5 أصول، 12 نافذة)، وسُجِّلت الـ12
مدخلاً (4 مرشّحين × 3 أهداف) في `experiment_registry/registry.json`
الحقيقي — **كلّها `مرفوضة`**، لا مرشّح واحد اجتاز `consistent_sign=True`.

**الأهمّ: `DPO_2_reversion`** — الذي أعطى profit_factor=1.14 (ربح صافٍ
موجب) في تجربة صاحب المشروع المباشرة على أصل واحد (BTC/1H) — **يُظهر IC
شبه معدوم فعلياً عبر الأصول الخمسة** (`close`≈+0.049، `high`≈-0.001،
`low`≈-0.001، لا أي منها معنوي إحصائياً). هذا مثال حيّ ودقيق تماماً لما
حذّرت منه الخطة عن الباكتيست الخام بلا محور تقييم: نتيجة تبدو رابحة على
عيّنة/أصل واحد قد تكون ضجيجاً بحتاً لا يصمد أمام أصول أخرى أو خطّ أساس
عشوائي — **بالضبط الفرق بين "بدا مربحاً" و"إشارة حقيقية"**.

الباقي (`CMO_14_reversion`، `TSI_5_13_5_momentum`، `DPO_14_reversion`)
أضعف من ذلك أو غير متّسق الاتجاه عبر النوافذ. لا شيء من هذه الدفعة يستحقّ
تفعيلاً في الإنتاج حتى الآن.

## ٦) الماسح الآلي — تقييم كل المرشّحين × كل الأهداف دفعة واحدة

In [ ]:
# @title
def scan_candidates(candidates, windows, targets=("close", "high", "low"),
                    feature_order=None, **eval_kwargs):
    """يُقيِّم كل مرشّح × كل هدف عبر evaluate_candidate (مع حارس
    clean_reg_target تلقائياً)، ويُرجع لوحة قيادة (leaderboard) مُرتَّبة —
    اتساق الإشارة أوّلاً، ثم قوة IC المطلقة. لا يتوقّف عند أوّل خطأ (يُسجَّل
    ويُكمل بقية المرشّحين) — مفيد خاصة لمرشّحين قد لا يحملهما كل dataset
    (مثل FUND_rate_extreme_position)."""
    rows = []
    for cand in candidates:
        predict_fn = make_candidate_predict_fn(cand, feature_order=feature_order)
        for target in targets:
            try:
                report = evaluate_candidate(predict_fn, target, windows, **eval_kwargs)
                rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                            "hypothesis": cand.get("hypothesis", ""),
                            "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                            "frac_significant": report["frac_significant"],
                            "consistent_sign": report["consistent_sign"],
                            "n_ok": report["n_ok"], "status": "ok"})
            except Exception as e:
                rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                            "status": f"error: {type(e).__name__}: {e}"})
    df = pd.DataFrame(rows)
    ok = df[df["status"] == "ok"].copy()
    if len(ok):
        ok["abs_mean_ic"] = ok["mean_ic"].abs()
        ok = ok.sort_values(["consistent_sign", "abs_mean_ic"], ascending=[False, False])
        df = pd.concat([ok.drop(columns="abs_mean_ic"), df[df["status"] != "ok"]], ignore_index=True)
    return df


leaderboard = scan_candidates(
    CANDIDATE_SIGNALS + EXPLORATORY_CANDIDATES + GENERATIVE_CANDIDATES + LEGACY_BACKTEST_CANDIDATES,
    windows, feature_order=FEATURE_ORDER)
pd.set_option("display.width", 160)
print(leaderboard.to_string(index=False))

### النتيجة الفعلية (تشغيل حقيقي — 5 أصول من `history_1d`، نفس بيانات H002)

أُجري هذا الماسح فعلياً على نفس بيانات H002 (5 أصول، 12 نافذة). **لا مرشّح
واحد من الـ17 حقّق `consistent_sign=True`** — أقوى النتائج (`NATR_14_neg` على
`close`، mean_ic=-0.234؛ `RSI_14_reversion` على `close`، mean_ic=+0.221)
غير متّسقة الاتجاه عبر النوافذ، مطابقة لصعوبة إيجاد إشارة يومية موثوقة التي
وثّقتها H001/H002 (سقف قريب من العملة المعدنية العادلة). **لم تُسجَّل أي
فرضية من هذا التشغيل في `experiment_registry`** — لا شيء عبر عتبة الثقة
(`consistent_sign=True` + معنوية كافية) يستحقّ تسجيلاً؛ راجع القسم ٨ أدناه
لكيفية التسجيل يدوياً حين يتوفّر مرشّح يستحقّه.

## ٧) البحث التركيبي الرخيص (`data_driven`/`genetic_search`) — بلا شبكة عصبية

مرشّح مركَّب: انحدار Ridge خطّي (`RidgeCV`، يُحسَب مغلقاً بلا حِقَب — أجزاء
من الثانية لكل نافذة) على مجموعة الميزات كلّها معاً، بدل مؤشر واحد. ليست
شبكة عصبية ولا تدريباً تكرارياً — أرخص بآلاف المرّات من تدريب نموذج NIG-TimeNet
لكل نافذة (راجع H002)، لكنها قد تلتقط تفاعلات بين الميزات لا يلتقطها أي
مرشّح فردي أعلاه.

In [ ]:
# @title
from sklearn.linear_model import RidgeCV

COMPOSITE_FEATURES = [c["feature"] for c in CANDIDATE_SIGNALS if c["feature"] != "MKT_ret_1"]
# ✅ extract_feature_matrix مُعرَّفة مرّة واحدة في قسم ٤ (إطار المرشّح) —
# يُعاد استخدامها هنا كما هي، ومن make_isolation_forest_predict_fn لاحقاً.


def make_ridge_composite_predict_fn(features, target, feature_order=None, alphas=(0.1, 1.0, 10.0, 100.0)):
    """يُدرِّب RidgeCV على train (مغلق، بلا حِقَب) متنبّئاً بـclean_reg_target
    لنفس target، ثم يُنبئ على test — نفس عقد predict_fn المُستخدَم مع
    evaluate_candidate."""
    def predict_fn(train, val, test):
        train_flat = concat_splits(train)
        Xtr = extract_feature_matrix(train_flat, features, feature_order=feature_order)
        ytr = clean_reg_target(train_flat, target)
        mu, sigma = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-9
        model = RidgeCV(alphas=alphas)
        model.fit((Xtr - mu) / sigma, ytr)
        Xte = extract_feature_matrix(test, features, feature_order=feature_order)
        return model.predict((Xte - mu) / sigma)
    return predict_fn


composite_rows = []
for target in ("close", "high", "low"):
    pf = make_ridge_composite_predict_fn(COMPOSITE_FEATURES, target, feature_order=FEATURE_ORDER)
    report = evaluate_candidate(pf, target, windows)
    composite_rows.append({"name": "ridge_composite_all_features", "target": target,
                           "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                           "frac_significant": report["frac_significant"],
                           "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})
print(pd.DataFrame(composite_rows).to_string(index=False))

### النتيجة الفعلية للمركَّب

على نفس البيانات: `close` mean_ic=+0.041 (غير معنوي)، `high` mean_ic=+0.209،
`low` mean_ic=+0.139 — **كلاهما `consistent_sign=False`**. لا تحسّن ذا شأن عن
أفضل مرشّح فردي، ولا اجتياز لعتبة القبول. **تنبيه مهم لمن يُعيد هذا الفحص:**
أوّل تشغيل لهذا المركَّب (قبل تطبيق حارس `clean_reg_target`) أعطى نتائج
خارقة زائفة (`high` mean_ic=+0.595، `consistent_sign=True`!) — وهذا بالضبط
ما كشف أثر مرجع "نفس النوع" الموثَّق أعلى الدفتر. أي نتيجة مستقبلية على
high/low أعلى من هذا المدى المتواضع تستحقّ فحصاً مضاعَفاً قبل تصديقها، لا
احتفالاً فورياً.

## ٨) المُشغّل الدفعي (Batch Runner) — تقييم متوازٍ + تسجيل تلقائي

طبقاً لـ"خطة بناء النظام — تسريع التجارب" في خطة المشروع، هذا يكمل الركيزتين
المتبقيتين من الأربع (الأخريان — واجهة موحّدة للفرضية، وذاكرة تخزين مؤقت —
مُلبّاتان أصلاً: `make_candidate_predict_fn` هو الواجهة الموحّدة، و`dataset`/
`windows` يُبنيان مرّة واحدة ويُعاد استخدامهما لكل مرشّح، بلا إعادة تحميل أو
إعادة حساب ميزات — بنفس روح `_checkpoint_fingerprint` في دفتر التحضير):

* **التوازي**: `imap_ordered`/`default_workers` من دفتر التحضير نفسه (لا
  إعادة كتابة) — خيوط، لا عمليات منفصلة (نفس مبرر التحضير: numpy/pandas
  تُحرِّر GIL).
* **معيار قبول موحّد** (`classify_result`): `مقبولة` فقط إن `consistent_sign=True`
  و`frac_significant` ≥ حدّ أدنى (0.34 افتراضياً — أي أكثر من ثلث النوافذ
  معنوية بنفس اتجاه المتوسط، دون تعسّف رقم أعلى بلا مبرّر). أقلّ من حدّ أدنى
  من النوافذ الناجحة (`n_ok`) → `قيد الاختبار` (لا حكم بعد). غير ذلك → `مرفوضة`.
* **تسجيل تلقائي**: كل (مرشّح × هدف) يُسجَّل في `experiment_registry` بمعرّف
  `SCAN_{اسم}_{هدف}` — **تمييز مهم**: هذه سجلّات آلية خفيفة من مسح دفعي، لا
  فرضيات مُنسَّقة يدوياً بعمق كـH001/H002 (تلك تبقى بمراجعة بشرية وتوثيق
  أوسع لكل شذوذ/ملاحظة). كلاهما في نفس السجلّ، والفائدة نفسها: يمنع اختبار
  نفس المرشّح مرتين لاحقاً بعد نسيان أنه فشل.

التسجيل اليدوي (بنمط H001/H002) يبقى الخيار الصحيح لأي فرضية تستحقّ تحليلاً
أعمق (شذوذ، مقارنة بخطّ أساس، تفسير سلوكي) — راجع تلك الدفاتر كمرجع للنمط.

In [ ]:
# @title
def classify_result(report, min_frac_significant=0.34, min_n_ok=5):
    """معيار قبول موحّد — نفس المنطق يُطبَّق بصرف النظر عن مصدر المرشّح
    (راجع "كيف تُقيَّم نتائج كل هذه الأدوات" في خطة المشروع). `n_ok` أقلّ من
    الحدّ الأدنى يعني عدد نوافذ ناجحة غير كافٍ للحكم أصلاً — لا "مرفوضة"
    مُتسرِّعة على دليل ضعيف."""
    if report.get("n_ok", 0) < min_n_ok:
        return "قيد الاختبار"
    if report.get("consistent_sign") and report.get("frac_significant", 0) >= min_frac_significant:
        return "مقبولة"
    return "مرفوضة"


def _batch_eval_job(job):
    cand, target, windows_, feature_order, eval_kwargs = job
    predict_fn = make_candidate_predict_fn(cand, feature_order=feature_order)
    try:
        report = evaluate_candidate(predict_fn, target, windows_, **eval_kwargs)
        return {"cand": cand, "target": target, "report": report, "status": "ok"}
    except Exception as e:
        return {"cand": cand, "target": target, "status": "error", "error": f"{type(e).__name__}: {e}"}


def run_batch_and_register(candidates, windows, targets=("close", "high", "low"), feature_order=None,
                           id_prefix="SCAN", max_workers=None, registry_path=None, **eval_kwargs):
    """المُشغّل الدفعي الكامل: يقيّم كل (مرشّح × هدف) بالتوازي عبر
    imap_ordered/default_workers (من دفتر التحضير)، يصنّف كل نتيجة عبر
    classify_result، ويسجّلها تلقائياً في experiment_registry. يُرجع
    (leaderboard, registered_ids) — الأولى للعرض السريع، والثانية لتتبّع ما
    كُتب فعلاً.

    ``registry_path``: مرّره (مثلاً tempfile) لتوجيه التسجيل بعيداً عن السجلّ
    الحقيقي — مفيد للاختبار الذاتي؛ اتركه ``None`` للمسار الافتراضي الحقيقي.
    """
    jobs = [(cand, target, windows, feature_order, eval_kwargs) for cand in candidates for target in targets]
    n_workers = max_workers or default_workers(len(jobs))

    rows, registered = [], []
    for res in imap_ordered(_batch_eval_job, jobs, max_workers=n_workers):
        cand, target = res["cand"], res["target"]
        if res["status"] == "error":
            rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                        "status": "error", "error": res["error"]})
            continue
        report = res["report"]
        status = classify_result(report)
        hyp_id = f"{id_prefix}_{cand['name']}_{target}"
        register_hypothesis(
            hyp_id=hyp_id,
            hypothesis=f"{cand.get('hypothesis', cand['name'])} (هدف: {target})",
            source=cand["track"],
            status=status,
            report={"per_window": report["per_window"], "mean_ic": report["mean_ic"],
                    "std_ic": report["std_ic"], "frac_significant": report["frac_significant"],
                    "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]},
            notes=(f"مُسجَّلة آلياً عبر run_batch_and_register. mean_ic={report['mean_ic']:.4f}, "
                  f"consistent_sign={report['consistent_sign']}, "
                  f"frac_significant={report['frac_significant']:.2f}."),
            registry_path=registry_path,
        )
        registered.append(hyp_id)
        rows.append({"name": cand["name"], "track": cand["track"], "target": target, "status": status,
                    "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                    "frac_significant": report["frac_significant"],
                    "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})

    df = pd.DataFrame(rows)
    ok = df[df["status"] != "error"].copy()
    if len(ok):
        ok["abs_mean_ic"] = ok["mean_ic"].abs()
        ok = ok.sort_values(["status", "abs_mean_ic"], ascending=[True, False])
        df = pd.concat([ok.drop(columns="abs_mean_ic"), df[df["status"] == "error"]], ignore_index=True)
    return df, registered


# مثال استخدام حقيقي (يكتب في experiment_registry الحقيقي — شغّله عمداً، لا تلقائياً):
# batch_leaderboard, registered_ids = run_batch_and_register(
#     CANDIDATE_SIGNALS + EXPLORATORY_CANDIDATES + GENERATIVE_CANDIDATES + LEGACY_BACKTEST_CANDIDATES,
#     windows, feature_order=FEATURE_ORDER)
# print(batch_leaderboard.to_string(index=False))
# print(f"سُجِّل {len(registered_ids)} مدخلاً: {registered_ids}")

### النتيجة الفعلية لتشغيل المُشغّل الدفعي الكامل (على السجلّ الحقيقي)

شُغِّل `run_batch_and_register` فعلياً (لا مثالاً معلَّقاً) على نفس بيانات
H002 (5 أصول، 12 نافذة) — **21 مرشّحاً قابلاً للتقييم × 3 أهداف = 63 مدخلاً
سُجِّلت في `experiment_registry/registry.json` الحقيقي** (المرشّح الثاني
والعشرون، `FUND_rate_extreme_position`، فشل بخطأ واضح على الأهداف الثلاثة
كما هو مصمَّم — `FUND_rate_z` غير مُفعَّلة في هذا الـdataset، فلم يُسجَّل).

**كل الـ64 مدخلاً (63 + `H002_nig_timenet_classification_head` سابقاً)
بحالة `مرفوضة`** — لا مرشّح واحد اجتاز `classify_result` (اتساق الاتجاه +
معنوية كافية) على هذه العيّنة من 5 أصول. هذا يُكمل **بوّابة خروج المرحلتين
١-٢** في خطة المشروع حرفياً ("توثيق كل نتيجة، مقبولة أو مرفوضة — لا حاجة
لنتيجة إيجابية للانتقال، فقط لدليل موثَّق"): بوّابة الخروج **اجتيزت
بالتوثيق**، لا بإيجاد إشارة. طبقاً لنص الخطة، هذا "مؤشّر مهم (لا نهائي) على
أن ميزات إضافية من نفس العائلة (OHLCV+مشتقّاتها المباشرة) لن تضيف كثيراً" —
يرفع أولوية إمّا (أ) توسيع عيّنة الأصول قبل الحكم النهائي (5 أصول عيّنة
صغيرة)، أو (ب) الانتقال للمرحلة ٣ في الخطة (Matrix Profile، SHAP
interactions، العنقدة — أدوات بلا قواعد مسبقة)، وليس تكرار مزيد من مرشّحين
من نفس العائلة. **قرار الانتقال يبقى صريحاً بيد صاحب المشروع، لا انزلاقاً
تلقائياً** (نفس مبدأ "الترتيب الزمني" في الخطة).

## ٩) المرحلة ٣ — أدوات اكتشاف بلا قواعد مسبقة (Matrix Profile + SHAP)

بقرار صريح من صاحب المشروع (بعد أن رفضت المرحلتان ١-٢ كل الفرضيات — ٧٦
مدخلاً موثَّقاً في السجلّ حتى الآن)، ننتقل للمرحلة ٣ في الخطة: أدوات لا
تفترض شكل الإشارة مسبقاً، بل تكتشف البنية من البيانات نفسها. كل أداة هنا
تُنتج مرشّحاً، لا إشارة مؤكَّدة — تمرّ عبر نفس محور IC + عُشر + خطّ أساس
عشوائي قبل أي قرار، مطابقةً لمبدأ "لا فرق في المعاملة بين نمط اكتشفته شجرة
قرار ونمط اقترحه حدسك" في الخطة.

### ٩-أ) Matrix Profile — تنبؤ بأقرب نافذة تاريخية مشابهة شكلياً

الأساس الرياضي لـMatrix Profile: بحث أقرب جار (k-NN) بمسافة إقليدية بعد
تطبيع-Z، بلا افتراض مسبق عن شكل الزخرفة (motif). محسوب هنا مباشرة (لا عبر
مكتبة `stumpy`) لأن بيانات هذا الدفتر مُقسَّمة مسبقاً لنوافذ منفصلة عبر
`rolling_splits`، لا سلسلة خام متصلة واحدة لكل أصل تصلح لمسح `stumpy`
التقليدي عبر حدود النوافذ — نفس النتيجة الرياضية (بحث أقرب جار
z-normalized Euclidean)، بلا افتعال حدود اصطناعية عبر دمج نوافذ منفصلة في
سلسلة واحدة. `target`-محدَّد كالمركَّب Ridge أعلاه (يحتاج عائد train
الفعلي كـ"مكتبة" للتنبؤ)، فيُقيَّم يدوياً لكل هدف على حدة، لا عبر
`scan_candidates` تلقائياً.

In [ ]:
# @title
def make_matrix_profile_predict_fn(target, feature="close", feature_order=None, k=3):
    """أساس Matrix Profile: بحث أقرب جار (k-NN) بمسافة إقليدية بعد تطبيع-Z
    — لكل نافذة اختبار، تُطبَّع نافذة `feature` (افتراضياً `close`)، تُقارَن
    بمكتبة نوافذ train كلّها، ويُتنبَّأ بمتوسط العائد الفعلي (`target`) الذي
    تلا أقرب k نافذة تاريخياً مشابهة شكلياً. راجع الشرح أعلاه لسبب الحساب
    المباشر بدل `stumpy.mass`."""
    def predict_fn(train, val, test):
        train_flat = concat_splits(train)
        train_series = extract_feature_series(train_flat, feature, feature_order=feature_order)
        train_targets = clean_reg_target(train_flat, target)
        test_series = extract_feature_series(test, feature, feature_order=feature_order)

        def znorm(a):
            mu = a.mean(axis=1, keepdims=True)
            sd = a.std(axis=1, keepdims=True) + 1e-9
            return (a - mu) / sd

        Ztr, Zte = znorm(train_series), znorm(test_series)
        preds = np.empty(len(Zte))
        for i in range(len(Zte)):
            d = np.linalg.norm(Ztr - Zte[i], axis=1)
            k_eff = min(k, len(d))
            nn_idx = np.argpartition(d, k_eff - 1)[:k_eff]
            preds[i] = train_targets[nn_idx].mean()
        return preds
    return predict_fn


matrix_profile_rows = []
for target in ("close", "high", "low"):
    pf = make_matrix_profile_predict_fn(target, feature="close", feature_order=FEATURE_ORDER, k=3)
    report = evaluate_candidate(pf, target, windows)
    matrix_profile_rows.append({"name": "matrix_profile_knn3_close_shape", "target": target,
                                "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                                "frac_significant": report["frac_significant"],
                                "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})
print(pd.DataFrame(matrix_profile_rows).to_string(index=False))

### النتيجة الفعلية لـMatrix Profile

على نفس بيانات H002 (5 أصول، 12 نافذة): `close` mean_ic=-0.019، `high`
mean_ic=+0.012، `low` mean_ic=-0.028 — **ضعيفة جداً وغير معنوية عملياً على
الثلاثة**، `consistent_sign=False`. أقرب k=3 نافذة تاريخية مشابهة شكلياً
لسعر الإغلاق وحده لا تحمل معلومة اتجاه مفيدة في هذا الإعداد — **مرفوضة**.
لم تُختبَر بعد: أشكال مبنية على أكثر من `close` (مثلاً نافذة الشمعة الكاملة
عبر عدّة ميزات معاً)، أو قيم `k` أخرى — خطوة تالية محتملة لا مُنفَّذة الآن.

### ٩-ب) SHAP interaction values — اكتشاف تفاعلات آلياً بلا تحديد يدوي

أداة اكتشاف لا تنبؤ (كما تنص الخطة): تُدرَّب غابة عشوائية ضحلة (عمق ٤) على
كل الميزات الخام معاً لنافذة `train` واحدة، ثم تُفحَص SHAP interaction
values — تكشف أي أزواج ميزات تتفاعل فعلياً في قرارات الشجرة آلياً، بخلاف
`VOLZ_x_BODY` أعلاه (تفاعل حُدِّد يدوياً بحدس سلوكي). **تُشغَّل مرّة واحدة
فقط** (على `train` النافذة الأولى، هدف `close`) لفصل "الاكتشاف" عن
"التحقّق" — الأزواج المُكتشَفة تُختبَر لاحقاً عبر كل النوافذ الـ12 كمرشّحين
عاديين (`kind="interaction"`)، لا على نفس بيانات الاكتشاف (يمنع تسرّباً
دائرياً بين مصدر الفرضية ومحكّ قبولها).

In [ ]:
# @title
!pip install -q shap 2>/dev/null


def discover_shap_interaction_pairs(train, target, feature_order, top_k=5, max_depth=4,
                                    n_estimators=200, random_state=42):
    """يُدرِّب RandomForestRegressor ضحلاً على متجه الميزات الكامل (آخر خطوة
    زمنية، عبر extract_feature_matrix)، يحسب SHAP interaction values
    (`shap.TreeExplainer`)، ويُرجع أقوى top_k أزواج ميزات (بمعزل عن القطر —
    الأثر الرئيسي المنفرد لكل ميزة، لا تفاعلاً) مرتّبة تنازلياً بمقدار
    التفاعل المطلق المتوسط."""
    from sklearn.ensemble import RandomForestRegressor
    import shap
    train_flat = concat_splits(train)
    X = extract_feature_matrix(train_flat, feature_order=feature_order)
    y = clean_reg_target(train_flat, target)
    model = RandomForestRegressor(max_depth=max_depth, n_estimators=n_estimators,
                                  random_state=random_state, n_jobs=-1).fit(X, y)
    explainer = shap.TreeExplainer(model)
    interaction_values = explainer.shap_interaction_values(X)
    mean_abs = np.abs(interaction_values).mean(axis=0)
    np.fill_diagonal(mean_abs, 0.0)
    F = len(feature_order)
    pairs = [(i, j, mean_abs[i, j]) for i in range(F) for j in range(i + 1, F)]
    pairs.sort(key=lambda t: t[2], reverse=True)
    return [(feature_order[i], feature_order[j], float(mag)) for i, j, mag in pairs[:top_k]]


shap_train, _shap_val, _shap_test = windows[0]
discovered_pairs = discover_shap_interaction_pairs(shap_train, "close", FEATURE_ORDER, top_k=5)
print("أقوى 5 أزواج تفاعل (SHAP interaction values، نافذة ١ فقط، هدف close):")
for a, b, mag in discovered_pairs:
    print(f"  {a} × {b}: {mag:.5f}")

SHAP_DISCOVERED_CANDIDATES = [
    {"name": f"SHAP_{a}_x_{b}", "track": "data_driven", "kind": "interaction",
     "feat_a": a, "feat_b": b, "op": "mul", "transform": None,
     "hypothesis": f"اكتشاف آلي عبر SHAP interaction values (غابة عشوائية ضحلة على train النافذة ١) — "
                   f"{a}×{b} من أقوى الأزواج تفاعلاً (مقدار≈{mag:.4f})"}
    for a, b, mag in discovered_pairs
]
print(f"\n{len(SHAP_DISCOVERED_CANDIDATES)} مرشّح تفاعل مُكتشَف آلياً — يُختبَر أدناه عبر كل النوافذ الـ12.")

shap_leaderboard = scan_candidates(SHAP_DISCOVERED_CANDIDATES, windows, feature_order=FEATURE_ORDER)
print(shap_leaderboard.to_string(index=False))

### النتيجة الفعلية لـSHAP interactions

على نفس بيانات H002: SHAP اكتشف آلياً (على `train` النافذة ١ فقط، هدف
`close`) أن `VOLZ_20` يتفاعل مع أربع ميزات أخرى أكثر من أي زوج آخر —
`NATR_14`، `BODY_ratio`، `VOLR_12_48`، `RET_3`، `STOCHk_14_3_3` — منطقي
سلوكياً (فورة حجم تُفسَّر بالتقلّب/شكل الشمعة/الزخم القصير معاً، لا بمعزل
عنها). عند اختبار الأزواج الخمسة عبر كل النوافذ الـ12: **لا شيء اجتاز
`consistent_sign=True`**، لكن `NATR_14×VOLZ_20` على `high` هو **أقرب نتيجة
لعتبة القبول في هذا الدفتر بأكمله** — `mean_ic=+0.225`، `frac_significant
=0.333` (تحت الحدّ الأدنى 0.34 بفارق ضئيل جداً)، رغم `consistent_sign
=False`. **مرفوضة رسمياً بمعيار `classify_result`، لكنها الأقرب من بين كل
ما اختُبر في هذا الدفتر** — تستحقّ تكراراً على عيّنة أصول أوسع قبل الحسم
النهائي، لا تفعيلاً فورياً. باقي الأزواج (`BODY_ratio`/`VOLR_12_48`/
`RET_3`/`STOCHk_14_3_3` × `VOLZ_20`) أضعف بوضوح.

### ٩-ج) أهمية ميزات مُجمَّعة (RF+GB+XGB) — من استكشاف سابق لصاحب المشروع

شارك صاحب المشروع دفتر استكشاف `xgboost.ipynb` قديماً خاصاً به: يُدرِّب
Random Forest + Gradient Boosting + XGBoost معاً على ~150 مؤشر خام
(تصنيف اتجاه الشمعة التالية)، ثم يطبع أهمّ 25 ميزة بمتوسط `feature_
importances_` الثلاثة موزونة بأوزان الدمج المُحسَّنة على validation.
**تقييم الطريقة الأصلية:** دقتها 51-54% وAUC 0.51-0.55 عبر كل تشغيلاتها —
ضعيفة جداً (قريبة من التخمين العشوائي)، **متّسقة تماماً مع كل ما وجدناه في
هذا المشروع بمنهجية مختلفة كلياً** (دليل مستقلّ إضافي على صعوبة إشارة
الاتجاه اليومي/الشمعة القادمة)، لكن التقييم نفسه ضعيف منهجياً: تقسيم
زمني واحد فقط (لا `rolling_splits`)، بلا تصحيح للاختبارات المتعددة، وأهمية
الميزات غير مُتحقَّق من ثباتها عبر فترات مختلفة.

**الفكرة القابلة لإعادة الاستخدام** (لا الأرقام، بل الآلية): إجماع أهمية
ميزات من عدّة نماذج مختلفة الطبيعة (أشجار مستقلّة/معزَّزة تدريجياً/معزَّزة
تدرّجياً بضبط أدقّ) أكثر متانة من نموذج واحد — نفس روح "غابة عشوائية ضحلة +
SHAP" أعلاه، لكن على مستوى **الميزة المفردة** لا التفاعل الثنائي، وبإجماع
ثلاثة نماذج بدل واحد. مُطبَّقة هنا بانضباط هذا الدفتر: انحدار لا تصنيف
(نتوافق مع `clean_reg_target` الصارم لا هدفاً ثنائياً)، اكتشاف مرّة واحدة
فقط على `train` النافذة الأولى (لا كل نافذة)، والميزات المُكتشَفة تُختبَر
لاحقاً عبر كل النوافذ كمرشّحين عاديين — نفس فصل "الاكتشاف عن التحقّق"
المُطبَّق مع SHAP.

In [ ]:
# @title
!pip install -q xgboost 2>/dev/null


def discover_ensemble_feature_ranking(train, target, feature_order, top_k=10, random_state=42):
    """إجماع أهمية ميزات من ثلاثة نماذج مختلفة الطبيعة (RandomForestRegressor
    + GradientBoostingRegressor + XGBRegressor)، بمتوسط `feature_importances_`
    الثلاثة بالتساوي. انحدار على `clean_reg_target` (لا تصنيف ثنائي كالأصل
    في `xgboost.ipynb`) ليتوافق مع صرامة هذا الدفتر. تُشغَّل مرّة واحدة فقط
    (على `train` النافذة الأولى) — نفس فصل الاكتشاف عن التحقّق المُطبَّق
    مع SHAP أعلاه."""
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from xgboost import XGBRegressor
    train_flat = concat_splits(train)
    X = extract_feature_matrix(train_flat, feature_order=feature_order)
    y = clean_reg_target(train_flat, target)
    rf = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=20,
                               random_state=random_state, n_jobs=-1).fit(X, y)
    gb = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.05,
                                   random_state=random_state).fit(X, y)
    xgbr = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=random_state,
                        n_jobs=-1, tree_method="hist", verbosity=0).fit(X, y)
    imp = (rf.feature_importances_ + gb.feature_importances_ + xgbr.feature_importances_) / 3.0
    order = np.argsort(imp)[::-1][:top_k]
    return [(feature_order[i], float(imp[i])) for i in order]


ens_train, _ens_val, _ens_test = windows[0]
ranked_features = discover_ensemble_feature_ranking(ens_train, "close", FEATURE_ORDER, top_k=10)
print("أقوى 10 ميزات (إجماع RF+GB+XGB، نافذة ١ فقط، هدف close):")
for name, imp in ranked_features:
    print(f"  {name}: {imp:.5f}")

ENSEMBLE_DISCOVERED_CANDIDATES = [
    {"name": f"ENSFI_{name}", "track": "data_driven", "feature": name, "transform": None,
     "hypothesis": f"اكتشاف آلي عبر إجماع أهمية ميزات RF+GB+XGB (train النافذة ١) — {name} من "
                   f"أقوى 10 ميزات (أهمية≈{imp:.4f})"}
    for name, imp in ranked_features
]
print(f"\n{len(ENSEMBLE_DISCOVERED_CANDIDATES)} مرشّح مُكتشَف آلياً — يُختبَر أدناه عبر كل النوافذ.")

ensemble_leaderboard = scan_candidates(ENSEMBLE_DISCOVERED_CANDIDATES, windows, feature_order=FEATURE_ORDER)
print(ensemble_leaderboard.to_string(index=False))

### النتيجة الفعلية لأهمية الميزات المُجمَّعة

على نفس بيانات H002: الإجماع (RF+GB+XGB على `train` النافذة ١، هدف
`close`) صنّف `VOLZ_20` أولاً بفارق واضح (أهمية 0.111 مقابل 0.068 للثانية،
`WICK_lower`) — **نفس الميزة التي اكتشفتها SHAP كمحور التفاعلات أعلاه،
بأداة مختلفة كلياً وبلا أي تنسيق بينهما** — تقارب مستقلّ يستحقّ الانتباه
حتى لو لم يترجم بعد لمرشّح مقبول. سُجِّلت الـ30 مدخلاً (10 ميزات × 3
أهداف) في `experiment_registry` الحقيقي، **كلّها `مرفوضة` رسمياً**، لكن
`RANGE_rel` على `high` نتيجة لافتة: `consistent_sign=True` (نادر في هذا
الدفتر) بـ`mean_ic=+0.220`، لكن `frac_significant=0.083` بعيد جداً عن
الحدّ الأدنى 0.34 — اتجاه ثابت لكن ضعيف الحدّة، لا يكفي للقبول. `NATR_14`
(0.234/0.141/0.119 على close/high/low) و`BBB_20_2.0` أيضاً من أقوى
النتائج رقمياً لكن `consistent_sign=False` لكليهما.

### ٩-د) اختبار تاريخي واسع (WIDEHIST) — إغلاق سؤال "العيّنة الصغيرة"

كل النتائج أعلاه (الأقسام ٦-٩) استخدمت `max_windows=12` فقط — أي ما يقارب
سنة واحدة من النوافذ المتحرّكة (بداية أوّل نافذة نوفمبر ٢٠٢١، آخرها مارس
٢٠٢٢). هذا فتح سؤالاً مُتكرِّراً منذ H002: هل النتائج السلبية (رفض كل
الفرضيات) بسبب ضعف حقيقي في الإشارات، أم بسبب عيّنة صغيرة جداً (12 نافذة
فقط، على 5 أصول مترابطة الحركة أصلاً)؟

**الاكتشاف**: نفس بيانات الـ5 أصول المخزَّنة محلياً (`history_1d`) تمتدّ
فعلياً لسنوات أطول بكثير مما استُخدم — SOLUSDT حتى نوفمبر ٢٠٢٠، والبقية
حتى ٢٠٢٤/٢٠٢٥ — بلا حاجة لأي جلب شبكي جديد (طلبات مباشرة لـ
`api.binance.com`/`fapi.binance.com` مرفوضة أصلاً بسياسة الشبكة هنا،
تأكَّد ذلك عبر `curl` مباشرة: `connect_rejected`/403).

رفع `max_windows` من 12 إلى 60 (بقيّة الإعدادات كما هي:
`test_span="30D"`, `initial_train_span="365D"`, `step="30D"`) أنتج **56
نافذة فعلية** تمتدّ من 2022-02-08 حتى 2026-09-14 — أي كامل التاريخ
المتاح فعلياً، لا استطالة تعسّفية.

**ما أُعيد اختباره على الـ56 نافذة**:
- كل مرشّحي الأقسام ٥-٧ (`CANDIDATE_SIGNALS` + `EXPLORATORY_CANDIDATES`
  + `GENERATIVE_CANDIDATES` + `LEGACY_BACKTEST_CANDIDATES` من ملحق ٣) —
  26 مرشّحاً × 3 أهداف.
- 5 أزواج تفاعل SHAP المُكتشَفة في القسم ٩-ب.
- مرشّح Matrix Profile (القسم ٩-أ).

**النتيجة**: **93 تجربة `WIDEHIST_*` مُسجَّلة، كلّها بحالة "مرفوضة"** —
ولا تجربة واحدة منها حقّقت `consistent_sign=True` عبر الـ56 نافذة (أعلى
قيمة `|mean_ic|` هي 0.149 لـ`VOLZ_volume_shock/high`، لكن بإشارة غير
ثابتة عبر النوافذ، تماماً كنمط كل التجارب السابقة في هذا المشروع).

هذا **يُغلق سؤال "العيّنة الصغيرة" نهائياً**: النتيجة السلبية ليست أثراً
لِـ12 نافذة أو لتاريخ قصير — هي نفسها عبر 56 نافذة تمتدّ نحو 4.5 سنوات من
كل التاريخ المتاح فعلياً لهذه الأصول الخمسة. أي فرضية مستقبلية تحتاج إمّا
بيانات أصول إضافية غير مترابطة، أو آلية اكتشاف مختلفة جذرياً — لا مجرّد
نافذة أطول على نفس الأصول.

### ٩-هـ) العنقدة غير المُشرَفة (K-means / HDBSCAN) — استكمال المرحلة ٣

آخر أداة من "المرحلة ٣" في خطة المشروع (راجع "العنقدة غير المُشرَفة" هناك):
تُجمِّع نوافذ شكل السعر (`close` بعد تطبيع-Z، نفس تحويل Matrix Profile أعلاه)
في عناقيد متشابهة الشكل على `train`، ثم تفحص: **هل عضوية عنقود معيّن ترتبط
بعائد مستقبلي مختلف عن البقية؟** التنبؤ لكل عيّنة اختبار = متوسط العائد
الفعلي لأعضاء نفس العنقود في `train` (بدل قيمة خام مباشرة كما في المرشّحين
السابقين). خوارزميتان مقارنتان عمداً:

* **K-means** (`k=5` ثابت): كل نقطة تنضمّ إجبارياً لأقرب مركز — خط أساس بسيط.
* **HDBSCAN**: لا يفترض عدد عناقيد مسبقاً، ويترك النقاط غير المتماسكة "ضجيجاً"
  (`label=-1`) بلا تصنيف بدل إجبارها على عنقود لا تنتمي إليه فعلياً — أنسب
  نظرياً لبيانات مالية نادراً ما تُشكِّل كتلاً كروية نظيفة (نقاط الضجيج تأخذ
  متوسط `train` العام بدل متوسط عنقود وهمي).

اختُبرت الخوارزميتان مباشرة على نطاق **WIDEHIST** (56 نافذة، كامل التاريخ
المتاح — القسم ٩-د أعلاه) بدل نطاق 12 نافذة الافتراضي لهذا الدفتر: بما أن
سؤال "هل الرفض بسبب عيّنة صغيرة؟" أُغلق نهائياً هناك، إعادة الاختبار على
نطاق أضعف إحصائياً كانت ستضيف عملاً بلا معلومة جديدة.

In [ ]:
# @title
def make_cluster_regime_predict_fn(target, feature="close", feature_order=None, algo="kmeans",
                                    n_clusters=5, min_cluster_size=None, random_state=42):
    # حلقة يدوية بنفس نمط Matrix Profile/Ridge composite أعلاه — يحتاج
    # target صراحةً فلا يلائم بناء builder(feature_order) العام في
    # make_candidate_predict_fn. يُدرَّب على train (تجميع + متوسط عائد كل
    # عنقود)، ثم يُسقِط test على نفس العناقيد.
    def predict_fn(train, val, test):
        train_flat = concat_splits(train)
        train_series = extract_feature_series(train_flat, feature, feature_order=feature_order)
        train_targets = clean_reg_target(train_flat, target)
        test_series = extract_feature_series(test, feature, feature_order=feature_order)

        def znorm(a):
            mu = a.mean(axis=1, keepdims=True)
            sd = a.std(axis=1, keepdims=True) + 1e-9
            return (a - mu) / sd

        Ztr, Zte = znorm(train_series), znorm(test_series)
        global_mean = train_targets.mean()

        if algo == "kmeans":
            from sklearn.cluster import KMeans
            k_eff = max(2, min(n_clusters, len(Ztr) // 5))
            model = KMeans(n_clusters=k_eff, random_state=random_state, n_init=10)
            train_labels = model.fit_predict(Ztr)
            test_labels = model.predict(Zte)
        elif algo == "hdbscan":
            import hdbscan
            mcs = min_cluster_size or max(5, len(Ztr) // 20)
            model = hdbscan.HDBSCAN(min_cluster_size=mcs, prediction_data=True)
            train_labels = model.fit_predict(Ztr)
            test_labels, _ = hdbscan.approximate_predict(model, Zte)
        else:
            raise ValueError(f"algo غير مدعوم: {algo}")

        # نقاط الضجيج (label=-1 في train) لا تُشكِّل عنقوداً حقيقياً — تُستبعَد
        # من قاموس المتوسطات، فيرث أي test تُسقَط عليها (أو على عنقود غير
        # موجود في train) متوسط train العام بدل متوسط وهمي.
        cluster_mean = {lbl: train_targets[train_labels == lbl].mean()
                        for lbl in np.unique(train_labels) if lbl != -1}
        return np.array([cluster_mean.get(lbl, global_mean) for lbl in test_labels])
    return predict_fn


cluster_rows = []
for algo, algo_name in (("kmeans", "KMeans5_close_shape"), ("hdbscan", "HDBSCAN_close_shape")):
    for target in ("close", "high", "low"):
        pf = make_cluster_regime_predict_fn(target, feature="close", feature_order=FEATURE_ORDER, algo=algo)
        report = evaluate_candidate(pf, target, windows)
        cluster_rows.append({"name": algo_name, "target": target,
                             "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                             "frac_significant": report["frac_significant"],
                             "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})
print(pd.DataFrame(cluster_rows).to_string(index=False))

### النتيجة الفعلية للعنقدة (تشغيل حقيقي — نطاق WIDEHIST، 56 نافذة)

**6 تجارب `WIDEHIST_Cluster_*` مُسجَّلة، كلّها `مرفوضة`**، ولا واحدة منها
`consistent_sign=True`:

| الخوارزمية | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| KMeans (k=5) | close | -0.037 | 0.071 | False |
| KMeans (k=5) | high | +0.042 | 0.071 | False |
| KMeans (k=5) | low | +0.038 | 0.143 | False |
| HDBSCAN | close | -0.067 | 0.089 | False |
| HDBSCAN | high | -0.024 | 0.107 | False |
| HDBSCAN | low | +0.006 | 0.143 | False |

أعلى `|mean_ic|` هو 0.067 فقط (HDBSCAN/close) — أضعف بكثير من أي مرشّح مقبول
في هذا الدفتر، وبإشارة متذبذبة تماماً عبر النوافذ (`consistent_sign=False`
للجميع، تماماً كنمط كل نتائج المرحلة ٣). لا حاجة لتفسير سلوكي بأثر رجعي هنا
(بوّابة الخروج المذكورة في خطة المشروع) لأن النتيجة أصلاً لا تحمل نمطاً
يستحقّ التفسير — لا فرق منهجي بين HDBSCAN (يترك الضجيج بلا تصنيف) وK-means
(يُجبر كل نقطة على الانضمام): كلاهما فشل بنفس الدرجة تقريباً، وهو دليل إضافي
(لا حاسم) على أن **شكل نافذة `close` وحدها** (بمعزل عن باقي الميزات) لا يحمل
معلومة تنبّئية كافية، بصرف النظر عن طريقة تجميعها.

**بهذا تكتمل المرحلة ٣ من الخطة (Matrix Profile + SHAP interactions +
العنقدة)** — كل أدواتها الثلاث اختُبرت على النطاق التاريخي الكامل المتاح
(56 نافذة)، وكلّها بلا استثناء رفضت كل الفرضيات المُختبَرة عبرها.

## ١٠) اختبار رخيص لفرضية "الترابط العابر للأصول" — قبل أي معمارية مشتركة

فكرة استُقبِلت من صاحب المشروع: بدل اعتبار الارتباط العالي بين الأصول
الخمسة عائقاً إحصائياً فقط (راجع ملاحظة H002)، **استغلاله** عبر معمارية
نموذج مشترك — نموذج فرعي لكل عملة ينتهي بطبقة تجميع مشتركة تُخرج متجهاً
بحجم عدد العملات، بحيث يستفيد التنبؤ بكل عملة من حركة بقيّة العملات في نفس
اللحظة. قبل الالتزام بتصميم معماري جديد يحتاج تدريباً كاملاً لتقييمه (تكلفة
حوسبة عالية بلا GPU)، هذا اختبار رخيص لا يحتاج شبكة عصبية إطلاقاً: هل
معلومة بسيطة عن الأصول الأخرى (متوسط عائدها في نفس اللحظة) تحمل أي ارتباط
حقيقي بعائد كل عملة على حدة؟ إن لم تحمل حتى أبسط صيغة لهذه المعلومة أي IC،
فهذا مؤشّر (لا حاسم) على أن معمارية أعقد ستُصارع لاستخراج شيء من عدم.

**ملاحظة مهمة**: ميزة "القوة النسبية لـBTC" تحديداً (`MKT_beta`) غير قابلة
للاختبار حالياً — بيانات BTC نفسها غير متوفّرة محلياً (فقط الأصول الخمسة
alt، وBinance محجوبة بسياسة الشبكة). لكن الفكرة الأعمّ — تبادل معلومات بين
الأصول الخمسة أنفسهم، بلا حاجة لـBTC — قابلة للاختبار الآن بالكامل ببيانات
موجودة فعلاً.

**التصميم**: لكل عيّنة اختبار لأصل A عند لحظة t، يُجمَع `RET_1`/`RET_6`
(معلومة متزامنة متاحة فعلياً وقت القرار، لا مستقبلية) لبقيّة الأصول B≠A
**عند نفس اللحظة t بالضبط** (محاذاة بالطابع الزمني الفعلي في `last_candles`،
لا بالترتيب)، ثم يُؤخَذ متوسطها كتنبّؤ. هذا يحتاج شكل بيانات خاصاً
(`rolling_splits(..., keep_asset_test_separate=True)`) بخلاف كل مرشّحي هذا
الدفتر — الشكل المُجمَّع الافتراضي (`keep_asset_test_separate=False`) لا
يُبقي أي أثر لحدود الأصول داخل `test`/`train` (تحقّق مباشر من مصدر `_take`
في خط الأنابيب)، فيستحيل معرفة أي الصفوف تخصّ أي عملة منه.

**قيد فعلي مهم اكتُشف أثناء التنفيذ**: بما أن الأصول الخمسة لا تشترك كلّها
في نفس المدى الزمني (SOLUSDT منذ 2020، والبقية منذ 2024/2025)، أغلب نوافذ
WIDEHIST المبكرة (٢٠٢٢-أوائل ٢٠٢٤) تحوي **أصلاً واحداً فقط** في `test` —
لا معنى لـ"عائد الأصول الأخرى" فيها فتُستبعَد تلقائياً (`evaluate_windows`
يتجاهل نوافذ بعيّنات صالحة أقل من الحدّ الأدنى، لا يُسقِط التقييم كلّه).
تداخل حقيقي بين الأصول (تراكب كامل 30/30 يوماً) يبدأ فعلياً من النافذة ٢٥
تقريباً (~2024) وحتى نهاية WIDEHIST.

In [ ]:
# @title
def make_cross_asset_predict_fn(feature="RET_1", feature_order=None, agg="mean"):
    # يحتاج test = {اسم_الأصل: قسم} (keep_asset_test_separate=True) — يبني
    # قاموس بحث لكل أصل آخر (الطابع الزمني -> قيمة feature)، ثم لكل عيّنة
    # في كل أصل يُتوسَّط قيمة بقيّة الأصول عند نفس الطابع الزمني بالضبط
    # (NaN إن لم يوجد أي أصل آخر بنفس اللحظة تماماً — تُستبعَد لاحقاً في
    # حساب IC، لا تُصفَّر).
    def predict_fn(train, val, test):
        asset_feat, asset_ts = {}, {}
        for name, split in test.items():
            asset_feat[name] = extract_feature_last_value(split, feature=feature, feature_order=feature_order)
            asset_ts[name] = np.asarray(split["last_candles"])[:, TS_COL]

        preds = []
        for name in test.keys():
            ts = asset_ts[name]
            n = len(ts)
            other_names = [o for o in test if o != name]
            other_lookup = {o: dict(zip(asset_ts[o].tolist(), asset_feat[o].tolist())) for o in other_names}
            out = np.full(n, np.nan)
            for i in range(n):
                t = ts[i]
                vals = [other_lookup[o][t] for o in other_names if t in other_lookup[o]]
                if vals:
                    out[i] = float(np.mean(vals)) if agg == "mean" else float(np.median(vals))
            preds.append(out)
        return np.concatenate(preds)
    return predict_fn


# ملاحظة تشغيل: يحتاج windows_sep = rolling_splits(dataset, ..., keep_asset_test_separate=True)
# (لا windows الافتراضية أعلاه — راجع الشرح فوق) و evaluate_candidate العام (يقبل test بأي شكل
# طالما predict_fn يتعامل معه بنفسه). النتائج الفعلية أدناه أُنتِجت على نطاق WIDEHIST (56 نافذة).

### النتيجة الفعلية (تشغيل حقيقي — نطاق WIDEHIST، 56 نافذة، keep_asset_test_separate=True)

**33 من 56 نافذة صالحة** (البقية أصل واحد فقط، مُستبعَدة تلقائياً كما هو
متوقَّع). **6 تجارب `WIDEHIST_CrossAsset_*` مُسجَّلة، كلّها `مرفوضة`**:

| الميزة | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| متوسط RET_1 للأصول الأخرى | close | +0.009 | 0.091 | False |
| متوسط RET_1 للأصول الأخرى | high | -0.037 | 0.182 | False |
| متوسط RET_1 للأصول الأخرى | low | +0.100 | 0.242 | False |
| متوسط RET_6 للأصول الأخرى | close | -0.038 | 0.061 | False |
| متوسط RET_6 للأصول الأخرى | high | -0.042 | 0.242 | False |
| متوسط RET_6 للأصول الأخرى | low | -0.002 | 0.242 | False |

أعلى `|mean_ic|` هو 0.0996 فقط (RET_1/low)، بإشارة متذبذبة (`consistent_sign
=False` للجميع) — نفس نمط كل نتائج المشروع حتى الآن. **الخلاصة لصالح فرضية
المعمارية المشتركة**: أبسط صيغة ممكنة لمعلومة عابرة للأصول (متوسط عائد
الأصول الأربعة الأخرى في نفس اللحظة) **لا تحمل ارتباطاً خطياً/رتبياً قابلاً
للاستغلال** مع عائد أي هدف. هذا **لا يستبعد نهائياً** أن معمارية عميقة قد
تكتشف تفاعلاً غير خطي أعقد (مثلاً: تفاعل مشروط بالتقلّب، أو نمط قيادة-تبعية
بين زوج أصول محدَّد بدل متوسط الأربعة كلّهم) — اختبار IC خطي/رتبي بسيط لا
يستطيع كشف ذلك بالتصميم. لكنه **يرفع عبء الإثبات** على أي استثمار في معمارية
معقّدة ومكلفة تدريباً: لا دليل أوّلي رخيص يدعمها بعد، والأصول الخمسة نفسها
صغيرة العدد ومترابطة أصلاً (سبيرمان زوجي ≈0.64 بين عوائدها، H002) — عدد
"إشارات مستقلّة" فعلي محدود جداً بصرف النظر عن تعقيد المعمارية.

## ١١) 🎯 أول فرضية مقبولة من هذا الإطار — عيّنة موسّعة (50 أصلاً، تفعيل `market_context`)

استُغِلّ توفّر مجلد `history_1d` على Drive بمئات العملات (لا 5 فقط) لبناء
عيّنة أوسع بكثير وأقلّ ترابطاً: **50 أصلاً حقيقياً** (الأصول الخمسة
الأصلية + BTCUSDT + 44 عملة راسخة متنوّعة — ETH، XRP، ADA، DOGE، DOT، LTC،
ATOM، AVAX، LINK، AAVE، وغيرها)، مع تفعيل `market_context.enabled=True`
صراحةً (يُصلح خلل `MKT_beta` الموثَّق في حاشية خطة المشروع) و`BTCUSDT`
كعملة مرجعية حقيقية. النتيجة: **94,961 عيّنة، 37 ميزة، 30 نافذة متحرّكة**
(بدل 12-56 نافذة على 5 أصول فقط سابقاً).

أُعيد اختبار كل المرشّحين القياسيين الـ26 × 3 أهداف (٧٥ تجربة `EXPANDED_*`
جديدة) — **و ظهرت لأول مرّة في تاريخ هذا المشروع بأكمله 4 نتائج `مقبولة`
رسمياً** (لا `قيد الاختبار` ولا `مرفوضة`):

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| `NATR_neg_vol` (`-NATR_14`) | high | **-0.197** | **93.3%** | **True (30/30)** |
| `RSI_vol_adjusted_reversion` (`-(RSI_14-50)/NATR_14`) | high | **-0.198** | **93.3%** | **True (30/30)** |
| `NATR_neg_vol` | low | **+0.164** | **96.7%** | **True (30/30)** |
| `RSI_vol_adjusted_reversion` | low | **+0.167** | **96.7%** | **True (30/30)** |

**تحقّق من المتانة**: فحصتُ الجدول الكامل لكل نافذة (30 نافذة) للمرشّح
`NATR_neg_vol/high` — **كل نافذة على حدة سالبة الإشارة بلا استثناء واحد**
(من -0.018 إلى -0.39)، والقيمة تشتدّ تدريجياً مع النوافذ الأحدث (حين يتوفّر
عدد أصول أكبر حيّة معاً، فيقلّ ضجيج القياس) — نمط متّسق مع أثر حقيقي
متزايد الوضوح مع القوة الإحصائية، لا صدفة ناتجة عن نافذة أو أصل شاذّ
واحد.

**⚠️ تحذير منهجي مهم قبل الاحتفال**: النتيجتان المقبولتان **ليستا
اكتشافين مستقلّين فعلياً** — `NATR_neg_vol` هي ببساطة `-NATR_14` (تقلّب
مُطبَّع)، و`RSI_vol_adjusted_reversion` هي `-(RSI_14-50)/NATR_14` (RSI
مقسوماً على نفس `NATR_14`). تقارب حجم الأثر شبه التام بينهما (-0.197 مقابل
-0.198) يشير إلى أن **`NATR_14` نفسه هو المحرِّك الأساسي لكلا الإشارتين**،
لا تأكيداً مستقلّاً من مؤشّرين مختلفين. الفرضية الفعلية الواحدة التي تستحقّ
التسجيل الرسمي: **"تقلّب مرتفع حالياً (NATR_14) يسبق ارتداداً في نطاق
الشمعة القادمة (`high` أدنى، `low` أعلى) — أي انكماش نطاق الحركة بعد فورة
تقلّب"** — تفسير سلوكي معقول (تفريغ التقلّب/استنفاد الزخم بعد حركة حادّة)،
لا نمط بلا معنى.

**الخطوة التالية (قرار صريح مطلوب من صاحب المشروع، لا تنفيذ تلقائي)**:
هذه أول نتيجة في المشروع بأكمله تجتاز بوّابة القبول رسمياً — تستحقّ تسجيلاً
كاملاً بمنهجية H001/H002 في `signal_evaluation_axis (3).ipynb` (فرضية H003)
مع اختبارات إضافية قبل الاعتماد عليها فعلياً: (أ) هل تصمد على تحويل رتيب
مختلف لتفكيك أثر `NATR_14` عن التطبيع بـRSI تحديداً؟ (ب) هل تصمد بعد تصحيح
الاختبارات المتعددة (~26 مرشّحاً × 3 أهداف = 78 اختباراً في هذا التشغيل
وحده)؟ (ج) هل تنتقل فعلاً لتحسين حقيقي في تدريب `NIG-TimeNet v2` الكامل
عند إضافتها كميزة، أم تبقى IC نظرياً بلا أثر عملي على النموذج؟

## ١٢) مرشّح للاختبار القادم — فركتالات الانعكاس (Williams Fractals)

يفحص هذا المرشّح سؤال H003 المفتوح (ب) من زاوية مستقلّة تماماً عن
`NATR_14`: هل "الارتداد بعد تطرّف محلي" أثر حقيقي أوسع من مجرّد تقلّب
مُطبَّع، أم خاص بـ`NATR_14`/`RSI` تحديداً؟ الفركتال (Bill Williams) يحدّد
نقاط تطرّف محلي مباشرة من `high`/`low` الخام — بلا أي علاقة رياضية بـ
`NATR_14` أو `RSI` — فإن أظهر `consistent_sign=True` أيضاً، هذا دليل
مستقلّ حقيقي على نفس فكرة H003 (لا تكراراً لها)؛ وإن لم يُظهر، هذا يُضيّق
التفسير نحو أن `NATR_14` تحديداً (لا "التطرّف المحلي" عموماً) هو المحرِّك.

**الصيغة المقترَحة (من صاحب المشروع)**:

```python
window = 5
df['fractal_high'] = (df['high'] == df['high'].rolling(window, center=True).max()).astype(int)
df['fractal_low'] = (df['low'] == df['low'].rolling(window, center=True).min()).astype(int)
```

**⚠️ عائقان يجب حلّهما قبل التشغيل الفعلي — موثَّقان هنا صراحةً بدل تنفيذ الصيغة كما وردت بلا تدقيق:**

1. **`high`/`low` مُستبعَدتان من `feature_order` افتراضياً.** `DEFAULT_CONFIG["exclude_from_features"] = ["open", "high", "low", "volume"]` في `crypto_data_pipeline_v6.ipynb` (السبب الموثَّق هناك: ترابط 0.99 مع `close` بعد التطبيع، تُغطّى معلومتهما بـ`RANGE_rel`/`BODY_ratio`/`WICK_*`). الحلّ **بسيط ولا يحتاج ميزة جديدة في خط الأنابيب**: `exclude_features` قابلة للتراجع الجزئي — يكفي بناء `dataset` القادم بـ`exclude_from_features=["open", "volume"]` (إبقاء `high`/`low` فقط، لا استعادة الأربعة) ليصبح كلٌّ من `extract_feature_series(split, "high", ...)` و`extract_feature_series(split, "low", ...)` صالحاً فوراً بنفس نمط `kind="series"` المستخدَم أصلاً لـCMO/TSI/DPO في "ملحق ٣".
2. **`center=True` تُعيد رسم الماضي (نفس فخّ DPO الموثَّق في خطة المشروع، قسم "تقطير الرؤية المتأخّرة").** فركتال عند الخطوة `t` يحتاج معرفة `high`/`low` حتى `t+window//2` — غير متاحة فعلياً عند اتخاذ القرار عند `t`. الصيغة أدناه **لا تستخدم الفركتال عند آخر خطوة** (undefined فعلياً)، بل عند **آخر خطوة "مؤكَّدة"** — أي `t = T-1-window//2` — التي أصبح فيها `window//2` من "المستقبل" اللازم لتأكيدها متاحاً فعلياً ضمن النافذة المُشاهَدة نفسها (نفس حلّ "تأخير شمعة واحدة" الذي أبقى نتائج DPO ممتازة في الخطة).

In [ ]:
# @title
def _fractal_reversal_signal(high_2d, low_2d, window=5):
    """فركتال ويليامز على آخر خطوة "مؤكَّدة" فقط (T-1-window//2)، لا آخر
    خطوة في النافذة — تفادياً لتسرّب معلومة مستقبلية عبر center=True (راجع
    الخلية النصية أعلاه). -1 = آخر تطرّف مؤكَّد كان قمّة (نتوقّع ارتداداً
    هبوطياً)، +1 = كان قاعاً (نتوقّع ارتداداً صعودياً)، 0 = لا تطرّف عندها."""
    if window % 2 == 0:
        raise ValueError("window يجب أن يكون فردياً (مركز واضح لكل جهة).")
    half = window // 2
    confirmed_idx = high_2d.shape[1] - 1 - half
    if confirmed_idx < half:
        raise ValueError(
            f"طول النافذة الزمنية ({high_2d.shape[1]}) أقصر من اللازم "
            f"لتأكيد فركتال بعرض {window} — لا نقطة زمنية صالحة."
        )
    out = np.zeros(high_2d.shape[0])
    for i in range(high_2d.shape[0]):
        h, l = high_2d[i], low_2d[i]
        lo, hi = confirmed_idx - half, confirmed_idx + half + 1
        is_high = h[confirmed_idx] == h[lo:hi].max()
        is_low = l[confirmed_idx] == l[lo:hi].min()
        if is_high and not is_low:
            out[i] = -1.0
        elif is_low and not is_high:
            out[i] = 1.0
    return out


def make_fractal_reversal_predict_fn(feature_order=None, window=5):
    """`kind="series"` مزدوج (high وlow معاً) — يفشل بخطأ صريح إن لم تحمل
    dataset الحالية high/low فعلياً (راجع العائق ١ أعلاه)، لا صمتاً بصفر."""
    def predict_fn(train, val, test):
        test_flat = concat_splits(test)
        fo = feature_order if feature_order is not None else test_flat.get("feature_order")
        if not fo or "high" not in fo or "low" not in fo:
            raise ValueError(
                "fractal_reversal يحتاج 'high' و'low' في feature_order — "
                "أعد بناء dataset بـ exclude_from_features=['open', 'volume'] "
                "(بدل القائمة الافتراضية التي تستبعد high/low أيضاً)."
            )
        high_2d = extract_feature_series(test_flat, "high", feature_order=fo)
        low_2d = extract_feature_series(test_flat, "low", feature_order=fo)
        return _fractal_reversal_signal(high_2d, low_2d, window=window)
    return predict_fn


FRACTAL_REVERSAL_CANDIDATE = {
    # kind="trained" مُعاد استخدامه هنا لآلية "builder(feature_order=...) →
    # predict_fn" فقط (نفس نقطة التفرّع في make_candidate_predict_fn) — لا
    # تدريب فعلياً، بل إغلاق (closure) على window الثابتة، تماماً كنمط
    # make_ridge_composite_predict_fn/make_isolation_forest_predict_fn.
    "name": "Fractal_reversal_w5", "track": "literature_mining", "kind": "trained",
    "builder": make_fractal_reversal_predict_fn,
    "hypothesis": (
        "تطرّف محلي مؤكَّد في high/low (فركتال ويليامز، عرض 5) يسبق ارتداداً "
        "— اختبار مستقلّ عن NATR_14/RSI لسؤال H003 المفتوح (ب): هل الارتداد "
        "بعد تطرّف أثر عام أم خاص بالتقلّب المُطبَّع تحديداً؟"
    ),
}
print("✅ Fractal_reversal_w5 جاهز — يحتاج dataset مبنيّاً بـ "
      "exclude_from_features=['open', 'volume'] (لا الافتراضي) ليُشغَّل فعلياً؛ "
      "لم يُشغَّل بعد على بيانات حقيقية في هذه الجلسة.")

## ١٤) اختبار ذاتي (بيانات تركيبية — بلا حاجة لـDrive)

يتحقّق من سلامة الوصلات (`clean_reg_target` يُزيل الأثر المصطنع فعلاً،
`scan_candidates` لا يتعطّل، الماسح يُرجع أعمدة اللوحة المتوقَّعة) على بيانات
عشوائية صغيرة — لا يثبت وجود إشارة حقيقية، فقط أن البنية تعمل.

In [ ]:
# @title
def run_discovery_lab_selftest():
    rng = np.random.default_rng(0)
    n_assets, n_per_asset, T, F = 3, 200, 8, len(FEATURE_ORDER) if 'FEATURE_ORDER' in globals() else 37
    feature_order = FEATURE_ORDER if 'FEATURE_ORDER' in globals() else [f"f{i}" for i in range(F)]
    n = n_assets * n_per_asset
    ts0 = pd.Timestamp("2022-01-01", tz="UTC")
    ts = pd.concat([pd.Series(pd.date_range(ts0, periods=n_per_asset, freq="1D"))
                    for _ in range(n_assets)], ignore_index=True)

    X = rng.normal(size=(n, T, F)).astype("float32")
    last_close = 100.0 * np.exp(rng.normal(scale=0.05, size=n).cumsum() / n_per_asset)
    body_idx = feature_order.index("BODY_ratio") if "BODY_ratio" in feature_order else 0
    # ✅ نزرع أثر مرجع "نفس النوع" عمداً: last_high يعتمد على BODY_ratio لا على
    #    حركة سعرية حقيقية — clean_reg_target يجب أن يُزيله، والهدف الخام (لو
    #    استُخدم بالخطأ) يجب أن يُظهره بوضوح.
    body_last = X[:, -1, body_idx]
    last_high = last_close * (1.0 + np.clip(-body_last, 0, None) * 0.05 + 1e-3)
    last_low = last_close * (1.0 - np.clip(body_last, 0, None) * 0.05 - 1e-3)
    future_high_max = last_close * (1.0 + rng.normal(scale=0.01, size=n))  # لا علاقة حقيقية بـbody_last
    future_low_min = last_close * (1.0 - np.abs(rng.normal(scale=0.01, size=n)))
    future_close = last_close * (1.0 + rng.normal(scale=0.01, size=n))

    y_high_reg_dirty = (future_high_max - last_high) / last_high  # مرجع "نفس النوع" (ملوَّث)
    y_low_reg_dirty = (future_low_min - last_low) / last_low
    y_close_reg = (future_close - last_close) / last_close

    last_candles = np.stack([last_high, last_low, last_close, ts.values.astype("int64"),
                             future_close, future_low_min, future_high_max], axis=1)
    flat = {"base_params": np.zeros((n, 2), "float32"), "last_candles": last_candles,
            "X_1D": X, "y": {"y_high_reg": y_high_reg_dirty, "y_low_reg": y_low_reg_dirty,
                             "y_close_reg": y_close_reg}}

    # ١) الحارس يُزيل الأثر المزروع فعلاً
    clean_high = clean_reg_target(flat, "high")
    from scipy.stats import spearmanr
    rho_dirty, _ = spearmanr(body_last, y_high_reg_dirty)
    rho_clean, _ = spearmanr(body_last, clean_high)
    assert abs(rho_dirty) > 0.3, f"الأثر المزروع ضعيف جداً للاختبار ({rho_dirty:.3f}) — أصلح البيانات التركيبية."
    assert abs(rho_clean) < abs(rho_dirty) / 3, (
        f"❌ clean_reg_target لم يُزل الأثر المزروع: dirty={rho_dirty:.3f} clean={rho_clean:.3f}")
    print(f"  ✅ clean_reg_target يُزيل أثر مرجع نفس النوع (dirty={rho_dirty:+.3f} → clean={rho_clean:+.3f})")

    # ٢) extract_feature_last_value / make_feature_predict_fn يعملان
    v = extract_feature_last_value(flat, feature_order[0], tf="1D", feature_order=feature_order)
    assert v.shape == (n,), "extract_feature_last_value: شكل خاطئ."
    predict_fn = make_feature_predict_fn(feature_order[0], transform=lambda x: -x,
                                         tf="1D", feature_order=feature_order)
    preds = predict_fn(flat, flat, flat)
    assert np.allclose(preds, -v), "make_feature_predict_fn: التحويل لم يُطبَّق بشكل صحيح."
    print("  ✅ extract_feature_last_value / make_feature_predict_fn تعملان بشكل صحيح")

    # ٢-ب) make_candidate_predict_fn: كل الأنواع الأربعة (feature/interaction/custom/trained)
    feat_b = feature_order[1] if len(feature_order) > 1 else feature_order[0]
    inter_cand = {"kind": "interaction", "feat_a": feature_order[0], "feat_b": feat_b, "op": "mul"}
    inter_fn = make_candidate_predict_fn(inter_cand, tf="1D", feature_order=feature_order)
    a = extract_feature_last_value(flat, feature_order[0], tf="1D", feature_order=feature_order)
    b = extract_feature_last_value(flat, feat_b, tf="1D", feature_order=feature_order)
    assert np.allclose(inter_fn(flat, flat, flat), a * b), "make_candidate_predict_fn: مسار التفاعل خاطئ."

    custom_cand = {"kind": "custom", "fn": lambda X_last, fo: X_last[:, 0] * 2.0}
    custom_fn = make_candidate_predict_fn(custom_cand, tf="1D", feature_order=feature_order)
    X_last_expected = extract_feature_matrix(flat, tf="1D", feature_order=feature_order)
    assert np.allclose(custom_fn(flat, flat, flat), X_last_expected[:, 0] * 2.0), (
        "make_candidate_predict_fn: مسار kind='custom' خاطئ.")

    trained_cand = {"kind": "trained", "builder": lambda feature_order: make_isolation_forest_predict_fn(
        [feature_order[0], feat_b], feature_order=feature_order, contamination=0.1)}
    trained_fn = make_candidate_predict_fn(trained_cand, feature_order=feature_order)
    trained_preds = trained_fn(flat, flat, flat)
    assert trained_preds.shape == (n,), "make_candidate_predict_fn: مسار kind='trained' أرجع شكلاً خاطئاً."

    series_cand = {"kind": "series", "feature": feature_order[0], "fn": lambda s: s[:, -1] * 3.0}
    series_fn = make_candidate_predict_fn(series_cand, tf="1D", feature_order=feature_order)
    series_full = extract_feature_series(flat, feature_order[0], tf="1D", feature_order=feature_order)
    assert series_full.shape == (n, T), "extract_feature_series: شكل خاطئ."
    assert np.allclose(series_fn(flat, flat, flat), series_full[:, -1] * 3.0), (
        "make_candidate_predict_fn: مسار kind='series' خاطئ.")
    print("  ✅ make_candidate_predict_fn يدعم الأنواع الخمسة (feature/interaction/custom/trained/series)")

    # ٣) scan_candidates يُرجع لوحة قيادة بالأعمدة المتوقَّعة، بلا انهيار
    windows_synth = [(flat, flat, flat)]
    tiny_candidates = [{"name": "f0", "track": "data_driven", "feature": feature_order[0], "transform": None}]
    board = scan_candidates(tiny_candidates, windows_synth, targets=("close", "high", "low"),
                            feature_order=feature_order, n_shuffles=20, min_samples=5)
    expected_cols = {"name", "track", "target", "status"}
    assert expected_cols.issubset(board.columns), f"أعمدة ناقصة في اللوحة: {board.columns.tolist()}"
    assert (board["status"] == "ok").all(), f"فشل تقييم بعض المرشّحين:\n{board}"
    print("  ✅ scan_candidates يُرجع لوحة قيادة سليمة بلا أخطاء")

    # ٤) classify_result: منطق حتمي بمعزل عن أي تدريب/عشوائية
    assert classify_result({"n_ok": 2, "consistent_sign": True, "frac_significant": 1.0}) == "قيد الاختبار", (
        "classify_result: n_ok قليل يجب أن يُرجع 'قيد الاختبار' بصرف النظر عن باقي الحقول")
    assert classify_result({"n_ok": 10, "consistent_sign": True, "frac_significant": 0.5}) == "مقبولة", (
        "classify_result: consistent_sign=True + frac_significant كافية يجب أن يُرجع 'مقبولة'")
    assert classify_result({"n_ok": 10, "consistent_sign": False, "frac_significant": 0.9}) == "مرفوضة", (
        "classify_result: consistent_sign=False يجب أن يُرجع 'مرفوضة' مهما كانت frac_significant")
    assert classify_result({"n_ok": 10, "consistent_sign": True, "frac_significant": 0.1}) == "مرفوضة", (
        "classify_result: frac_significant دون الحدّ الأدنى يجب أن يُرجع 'مرفوضة'")
    print("  ✅ classify_result يُطبِّق معيار القبول الموحّد بشكل صحيح (٤ حالات)")

    # ٥) run_batch_and_register: تسجيل فعلي إلى ملف مؤقّت (لا experiment_registry الحقيقي إطلاقاً)
    import tempfile, json as _json
    from pathlib import Path
    tmp_registry = Path(tempfile.mkdtemp()) / "registry_selftest.json"
    batch_board, registered_ids = run_batch_and_register(
        tiny_candidates, windows_synth, targets=("close", "high", "low"), feature_order=feature_order,
        id_prefix="SELFTEST", max_workers=2, registry_path=tmp_registry, n_shuffles=20, min_samples=5)
    assert len(registered_ids) == 3, f"يُتوقَّع تسجيل 3 (مرشّح واحد × 3 أهداف)، وُجد {len(registered_ids)}"
    assert tmp_registry.exists(), "run_batch_and_register: لم يُكتَب ملف السجلّ المؤقّت إطلاقاً"
    entries = _json.loads(tmp_registry.read_text(encoding="utf-8"))
    assert {e["id"] for e in entries} == set(registered_ids), "معرّفات السجلّ المكتوبة لا تطابق registered_ids"
    assert all(e["status"] in REGISTRY_STATUSES for e in entries), "حالة غير صالحة في سجلّ مكتوب فعلياً"
    assert all(e["id"].startswith("SELFTEST_") for e in entries), "id_prefix لم يُطبَّق على المعرّفات"
    print(f"  ✅ run_batch_and_register يقيّم بالتوازي (imap_ordered/default_workers) ويسجّل فعلياً "
          f"في ملف JSON مستقلّ ({len(registered_ids)} مدخلات، registry_path مُخصَّص لا الحقيقي)")

    print("✅ نجحت كل اختبارات مختبر بحث الإشارات الذاتية.")


run_discovery_lab_selftest()